In [ ]:
%reload_ext autoreload
%autoreload 2
from importlib import reload

import os
import sys
import pickle
import logging
import warnings
import numpy as np
import astropy as ap
import scipy as sp
import scipy.stats
import matplotlib as mpl
import matplotlib.pyplot as plt

import h5py
import tqdm.notebook as tqdm

import kalepy as kale
import kalepy.utils
import kalepy.plot

import holodeck as holo
import holodeck.sams
import holodeck.gravwaves
from holodeck import cosmo, utils, plot, discrete, sams, host_relations, _PATH_DATA
from holodeck.constants import MSOL, PC, YR, MPC, GYR, SPLC, SCHW, NWTG
from pathlib import Path

# Silence annoying numpy errors
np.seterr(divide='ignore', invalid='ignore', over='ignore')
warnings.filterwarnings("ignore", category=UserWarning)

# Plotting settings
mpl.rc('font', **{'family': 'serif', 'sans-serif': ['Times'], 'size': 15})
mpl.rc('lines', solid_capstyle='round')
mpl.rc('mathtext', fontset='cm')
plt.rcParams.update({'grid.alpha': 0.5})
mpl.style.use('default')   # avoid dark backgrounds from dark theme vscode

log = holo.log
log.setLevel(logging.INFO)


# ---- Define filepath containing simulation galaxy merger data files ----#
# ---- (if using files not in _PATH_DATA) ---- #
_HOME_PATH = Path('~/').expanduser()
#p = os.path.join(_HOME_PATH, 'cosmo_sim_merger_data')
p = os.path.join(_HOME_PATH, 'nanograv/gensams')
#p = os.path.join(_HOME_PATH, 'holodeck/holodeck/data')
if os.path.exists(p):
    _SIM_MERGER_PATH = p
else:
    p = os.path.join(_HOME_PATH, 'nanograv/cosmo_sim_merger_data')
    if os.path.exists(p):
        _SIM_MERGER_PATH = p
    else:
        _SIM_MERGER_PATH = _PATH_DATA
#_SIM_MERGER_PATH = _PATH_DATA
print(f"{_SIM_MERGER_PATH=}")
# ------------------------------------------------------------------------ #


In [ ]:
from compare_sams import load_sams_from_pkl,calc_sam_dadt_from_pkl,plot_dadt,calc_aGW_for_Fixed_Time_2PL,calc_cumulative_thard, calc_total_tau_inner

In [ ]:
def calc_hctot(gwb_sam):
    return np.sqrt( np.sum(gwb_sam[0]**2,axis=2) + gwb_sam[1]**2 )

def load_sam_data(nloud=1, nreals=10, nfreqs=40, gpf_flag=False, tau=1.0,
                  data_dir=_SIM_MERGER_PATH, subdir=None, 
                  fname_type='manual_moddefs'):

    if fname_type=='manual_moddefs':
        samtype = 'gpf' if gpf_flag else 'gmr'
    
        # ---- Unpickle SAM data
        ## right now these files are just stored in the same directory as the notebooks
        ## should move to data dir once things have stabilized a bit
        #sam_pkl_fname=f'sam_nfreqs{NFREQS}_nreals{NREALS}_nloud{NLOUD}_tau{TAU}_{samtype}.pkl'
        #sam_pkl_fname=f'sam_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_model_type{model_type}_{samtype}.pkl'
        sam_pkl_fname=f'test_sam_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_manual_moddefs_{samtype}_tau{tau}.pkl'

    elif fname_type=='old_new_mods_compare':
        sam_pkl_fname = f'test_sam_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_old_new_mods_compare_tau{tau}.pkl'
    elif 'new_hardening' in fname_type:
        sam_pkl_fname = f'test_sam_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_{fname_type}.pkl'
    else:
        raise ValueError(f'{fname_type=} not defined.')
    
    if subdir is not None:
        fpath = '/'.join((data_dir, subdir, sam_pkl_fname))
    else:
        fpath = '/'.join((data_dir, sam_pkl_fname))
    with open(fpath, "rb") as f:
        print(f'unpickling SAM data: {sam_pkl_fname}')
        sams = pickle.load(f)
        #sam, hard, gwb_new_sam, gwb_sam, freqs, freqs_edges = sam_data
        #sam, hard, gwb_sam, MODEL_PARS = sam_data
   
    #return sam, hard, gwb_new_sam, gwb_sam, freqs, freqs_edges
    #return sam, hard, gwb_sam, MODEL_PARS
    return sams
    

In [ ]:
def gwb_amps(sams=None, dpops=None, fname='compare_gwb_amps', colors=None, lbl_extra=None,
             color_hardmods=False, gpf_flags=None):
    
    fig, axs = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=[10,10])

    freqs_to_plot = np.array([1/(10*YR), 1/(3*YR), 1/YR])
    
    for i,f in enumerate(freqs_to_plot):
 
        if i==freqs_to_plot.size-1:
            axs[i].set(xlabel=r'$\log_{10}(A_\mathrm{yr})$')
        axs[i].set(ylabel='Probability Density')
        axs[i].grid(alpha=0.2)
        
        if sams is not None:
            for k,s in enumerate(sams):
                freqs_sam = s.PARS['freqs']
                idx_sam = np.where(np.abs(freqs_sam-f)==np.abs(freqs_sam-f).min())[0]
                hctot = calc_hctot(s.gwb_sam)
                amp_sam = hctot[idx_sam,:].flatten()
                
                lw = 1.5
                col = 'k'
                ls = '-'
                if s.model_type == 'old':
                    lw=1.5
                    col='k'
                    ls='-'
                elif s.model_type == 'old_rc100':
                    lw=2.5
                    col='k'
                    ls=':' if dpops is None else '-'
                elif s.model_type == 'ph15':
                    lw=2.5 if dpops is None else 2
                    col='darkred' if dpops is None else 'k'
                    ls='-.'
                elif s.model_type == 'old_2s':
                    lw=2
                    col='k'
                    ls='--'
                elif s.model_type == 'astr_rc100':
                    lw=2.5 if dpops is None else 3
                    col='r' if dpops is None else 'k'
                    ls='-'
                elif s.model_type == 'astr_nuo0':
                    lw=2
                    col='darkgray'
                    ls='-.'
                elif s.model_type == 'astr':
                    lw=2
                    col='darkgray'
                    ls='-' if dpops is None else ':'
                else:
                    lw=2
                    col='darkgray'
                    #raise ValueError(f"havent defined plot style for {s.model_type=}")

                lbl = s.model_type
                if lbl_extra is not None:
                    if len(lbl_extra) == len(sams):
                        lbl += lbl_extra[k]
                gpf_lbls = [' (GMR)', ' (GPF)']
                if len(gpf_flags) == len(sams):
                    lbl += gpf_lbls[gpf_flags[k]] 

                if colors is not None and len(colors)==len(sams):
                    thiscol = colors[k]
                else:
                    thiscol = col
                kale.dist1d(np.log10(amp_sam), density=True, hist=False, confidence=False, carpet=False, 
                            lw=lw, ls=ls, color=thiscol, label=lbl, ax=axs[i])
                
        axs[i].set(title=f"Comparison of GWB amplitudes at f={f*YR:.1g} [1/YR]")
        axs[i].legend(fontsize=9)

        ## new hardening model type = 0:
        if 'new_hard' in fname:
            plt.suptitle(f"{fname}\n"
                         +f"tout={s.PARS['hard_outer_time']:.2g}, "
                         +f"rch={s.PARS['hard_rchar']:.2g}pc, "
                         +f"nuin={s.PARS['hard_gamma_inner']:.2g}, "
                         +f"r9={s.PARS['hard_r_gw_crit_9']:.2g}{s.PARS['hard_gw_crit_units']}, "
                         +f"alph={s.PARS['hard_alpha_gw_crit']:.2g}")
        else:
            plt.suptitle(fname)
    
    fig.savefig(f'{fname}_nloud{NLOUD}_nreals{NREALS}.png')
    #plt.show()

In [ ]:
def __plot_gwb(fobs, gwb, hc_ss=None, bglabel=None, sslabel=None, **kwargs):
    xx = fobs * YR
    fig, ax = figax(
        xlabel=LABEL_GW_FREQUENCY_YR,
        ylabel=LABEL_CHARACTERISTIC_STRAIN
    )
    if(hc_ss is not None):
        draw_ss_and_gwb(ax, xx, hc_ss, gwb, sslabel=sslabel,
                        bglabel=bglabel, **kwargs)
    else:
        draw_gwb(ax, xx, gwb, **kwargs)
    _twin_hz(ax)
    return fig

def __draw_gwb(ax, xx, gwb, nsamp=10, color=None, label=None, ls=None, lw=None,
               alpha=0.25, **kwargs):
    if color is None:
        color = ax._get_lines.get_next_color()
    if ls is None:
        ls = '-'
    if lw is None: 
        lw=1.0
    kw_plot = kwargs.pop('plot', {})
    kw_plot.setdefault('color', color)
    #print(f"in __draw_gwb() call:{lw=} {ls=} {alpha=}")    
    hh = __draw_med_conf(ax, xx, gwb, plot=kw_plot, label=label, lw=lw, ls=ls, **kwargs)
    if (nsamp is not None) and (nsamp > 0):
        nsamp_max = gwb.shape[1]
        idx = np.random.choice(nsamp_max, np.min([nsamp, nsamp_max]), replace=False)
        for ii in idx:
            ax.plot(xx, gwb[:, ii], color=color, alpha=alpha, lw=lw, ls=ls)

    return hh

def __draw_med_conf(ax, xx, vals, fracs=[0.50, 0.90], weights=None, plot={}, 
                    fill={}, filter=False, label=None, lw=1.0, ls='-'):
    #plot.setdefault('alpha', 0.75)
    #fill.setdefault('alpha', 0.2)
    plot.setdefault('alpha', 0.9)
    fill.setdefault('alpha', 0.1)
    percs = np.atleast_1d(fracs)
    assert np.all((0.0 <= percs) & (percs <= 1.0))

    # center the target percentages into pairs around 50%, e.g.  68 ==> [16,84]
    inter_percs = [[0.5-pp/2, 0.5+pp/2] for pp in percs]
    # Add the median value (50%)
    inter_percs = [0.5, ] + np.concatenate(inter_percs).tolist()
    # Get percentiles; they go along the last axis
    if filter:
        rv = [
            kale.utils.quantiles(vv[vv > 0.0], percs=inter_percs, weights=weights)
            for vv in vals
        ]
        rv = np.asarray(rv)
    else:
        rv = kale.utils.quantiles(vals, percs=inter_percs, weights=weights, axis=-1)

    med, *conf = rv.T
    # plot median
    hh, = ax.plot(xx, med, **plot, lw=lw, ls=ls, label=label)

    # Reshape confidence intervals to nice plotting shape
    # 2*P, X ==> (P, 2, X)
    conf = np.array(conf).reshape(len(percs), 2, xx.size)

    kw = dict(color=hh.get_color())
    kw.update(fill)
    fill = kw

    # plot each confidence interval
    for lo, hi in conf:
        gg = ax.fill_between(xx, lo, hi, **fill)

    return (hh, gg)

def invyr2hz(invyr):
    return invyr / YR
def hz2invyr(hz):
    return hz * YR
    
def compare_gwb_sim_vs_sam(sams, dpops, var_type=None, fid_value=None, gpf_flags=None, fpath='', 
                           save=True, fname_extra='', show_title=True, cmap_arr=None,
                           linestyles=None, ylim=(2.0e-17,2.0e-14), #ylim=(2.0e-16,8.0e-15), 
                           colors=None, lbl_extra=None, sam_colors=None,sam_lbls=None,
                           NLOUD=None, NREALS=None, TAU=None):

    valid_models = 0
    for s in sams:
        if s.gwb_sam is not None:
            valid_models += 1
    if valid_models == 0:
        log.warning("No elements in sam data had a valid gwb. Nothing to plot.")
        return
        
    LABEL_GW_FREQUENCY_YR = r"GW Frequency $[\mathrm{yr}^{-1}]$"
    LABEL_GW_FREQUENCY_HZ = r"GW Frequency $[\mathrm{Hz}]$"
    LABEL_GW_FREQUENCY_NHZ = r"GW Frequency $[\mathrm{nHz}]$"
    LABEL_SEPARATION_PC = r"Binary Separation $[\mathrm{pc}]$"
    LABEL_CHARACTERISTIC_STRAIN = r"GW Characteristic Strain"
    LABEL_HARDENING_TIME = r"Hardening Time $[\mathrm{Gyr}]$"
    LABEL_CLC0 = r"$C_\ell / C_0$"

    freq_min = np.min(sams[0].PARS['freqs'])*0.95
    freq_max = np.max(sams[0].PARS['freqs'])*1.05
    print(f"{freq_min=} {freq_max=}")
    _freqs, _freqs_edges = utils.pta_freqs()
    print(f"{_freqs.min()=} {_freqs.max()=}")
    print(f"{_freqs_edges.min()=} {_freqs_edges.max()=}")
    
    fig, ax = plot.figax(
        xlabel=LABEL_GW_FREQUENCY_YR,
        ylabel=LABEL_CHARACTERISTIC_STRAIN,
        xlim=(freq_min*YR,freq_max*YR),
        ylim=ylim,
        figsize=(5.5,4.5)
        #figsize=(5,4)
    )
    secax = ax.secondary_xaxis('top', functions=(invyr2hz, hz2invyr),xlabel=LABEL_GW_FREQUENCY_HZ)

    if cmap_arr is None:
        cm = plot._get_cmap('viridis')
        colors = cm(np.linspace(0, 1, len(sams)))
        #cm = plot._get_cmap('tab20b')
        #cm2 = plot._get_cmap('tab20c')
        #cm3 = plot._get_cmap('tab20')
        #if len(sams)<=14:
        #    colors= np.vstack([cm(np.arange(1,20,4)),
        #                      cm2(np.arange(0,16,4)),
        #                      cm3(np.array([6,12,16,18,10]))]).reshape(14,4)
        #elif len(sams)>14 and len(sams)<=60:
        #    colors= np.vstack(cm(np.arange(20)),
        #                      cm2(np.arange(20)),
        #                      cm3(np.arange(20))).reshape(60,4)
        #else:
        #    raise ValueError()
        #cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples', 'GnBu',
        #            'RdPu', 'YlGnBu', 'YlGn','PuBuGn', 'OrRd', 'PuRd', 'YlOrRd', 'BuPu',
        #            'PuBu', 'YlGnBu_r']*5        
        #cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples',
        #            'Greys', 'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
        #            'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']*2
    else:
        if len(cmap_arr) != len(sams):
            raise ValueError("len(cmap_arr) must match len(sams) if not None")
    colors_list = []
    
    frac = 0.50
    
    print(f"{len(sams)=}")
    if len(sams) > 0:
        sam_freqs = sams[0].PARS['freqs']
        xx = sam_freqs * YR
        #if sam_colors is None: 
        #    sam_colors = ['k']*len(sams)

        idx_fiducial = None
        for n,s in enumerate(sams):

            if cmap_arr is not None:
                cm = plot._get_cmap(cmap_arr[n])
                #colors = cmap(np.linspace(0.3, 1, max_to_plot+1))
                colors_list.append(cm(0.8))
            else:
                colors_list.append(colors[n])
                
            #print(f"{n=} {s.model_type=}")
            if s.gwb_sam is None:
                print("no plot generated for sam with invalid params:")
                #print([v for v in s.PARS.values()])
                continue
            
            if var_type is not None and fid_value is not None:
                if var_type=='hard_r_gw_crit_9' or var_type=='hard_rchar_9':
                    par = np.log10(s.PARS[var_type])
                else:
                    par = s.PARS[var_type]
                if np.abs(fid_value-par) < 1.0e-6:
                    idx_fiducial = n

            #print(f"{n=} {var_type=} {fid_value=} {s.PARS[var_type]=} {idx_fiducial=}")
            
            if s.model_type == 'old':
                lw=1.5
                col='k'
                ls='-'
            elif s.model_type == 'old_rc100':
                lw=2.5
                col='k'
                ls=':' if len(dpops)==0 else '-'
            elif s.model_type == 'ph15':
                lw=2.5 if len(dpops)==0 else 2
                col='darkred' if len(dpops)==0 else 'k'
                ls='-.'
            elif s.model_type == 'old_2s':
                lw=2
                col='k'
                ls='--'
            elif s.model_type == 'astr_rc100':
                lw=2.5 if len(dpops)==0 else 3
                col='r' if len(dpops)==0 else 'k'
                ls='-'
            elif s.model_type == 'astr_nuo0':
                lw=2
                col='darkgray'
                ls='-.'
            elif s.model_type == 'astr':
                lw=2
                col='darkgray'
                ls='-' if len(dpops)==0 else ':'
            else:
                lw=2
                col='darkgray'
                ls='-'
                #raise ValueError(f"havent defined plot style for {s.model_type=}")

            if sam_lbls is not None and len(sam_lbls)==len(sams):
                lbl = sam_lbls[n]
            else:
                #lbl = s.model_type
                lbl = ''
                if lbl_extra is not None:
                    if len(lbl_extra) == len(sams):
                        lbl += lbl_extra[n]

                #gpf_lbls = [' (GMR)', ' (GPF)']
                #if len(gpf_flags) == len(sams):
                #    lbl += gpf_lbls[gpf_flags[n]] 
                #    ls = '--' if gpf_flags[n] else '-'

            if colors_list is not None and len(colors_list)>=len(sams):
                thiscol = colors_list[n]
            else:
                thiscol = col
            lw = (len(sams)-n)*0.2 + 0.8
            if sam_lbls is not None:
                print(f"{n=} {sam_lbls[n]=}")
                lw += 0.8
                _al = 0.6
                if linestyles is not None and len(linestyles)==len(sams):
                    ls = linestyles[n]
                else:
                    if n==0: 
                        ls='-'
                    else:
                        ls='--'
            else:
                if n==idx_fiducial:
                    ls = '-'
                    _al = 0.5
                    lw += 2
                else:
                    ls = '--'
                    _al = 0.05
            
            s_hctot = calc_hctot(s.gwb_sam)
            print(f"{s_hctot.min()=:.4g} {s_hctot.max()=:.4g}")
            print(f"before __draw_gwb() call: {n=} {lw=} {ls=} {_al=}")
            #__draw_gwb(ax, xx, s_hctot, nsamp=0, color=sam_colors[n], label=lbl,
            if n==idx_fiducial:
                __draw_gwb(ax, xx, s_hctot, nsamp=0, color=colors_list[n], label=lbl,
                           lw=lw, ls=ls, alpha=_al, fracs=[frac])
            else:
                __draw_gwb(ax, xx, s_hctot, nsamp=0, color=colors_list[n], label=lbl,
                           lw=lw, ls=ls, alpha=_al, fracs=[frac])

    if sam_lbls is not None:
        legend_title='Model type'
    else:
        if var_type=='hard_r_gw_crit_9':
            legend_title=r'a$_{\rm GW,9}$ [${\rm R_g}$]'
        elif var_type=='hard_alpha_gw_crit':
            legend_title=r'$\alpha_{\rm GW}$'
        elif var_type=='hard_beta_gw_crit':
            legend_title=r'$\beta_{\rm GW}$'
        elif var_type=='hard_rchar_9':
            legend_title=r'a$_{\rm char,9}$ [pc]'
        elif var_type=='hard_nu_inner':
            legend_title=r'$\nu_{\rm inner}$'
        elif var_type=='hard_outer_time':
            legend_title=r'$\tau_{\rm out}$ [Gyr]'
        else: 
            legend_title=""  
        
    plt.legend(loc='lower left',fontsize=10,title=legend_title) 

    if show_title:
        if 'new_hard' in fname_extra or 'newhard' in fname_extra:
            suptitl = (f"{fname_extra}\n"
                         +f"tout={s.PARS['hard_outer_time']:.2g}, "
                         +f"rch9={s.PARS['hard_rchar_9']:.2g}pc, "
                         +f"alphch={s.PARS['hard_alpha_char']:.2g}, "       
                         +f"rgw9={s.PARS['hard_r_gw_crit_9']:.2g}{s.PARS['hard_gw_crit_units']}, "
                         +f"alphgw={s.PARS['hard_alpha_gw_crit']:.2g}, ")
            if s.hard._inner_model_type == 0:
                suptitl += f"nuin={s.PARS['hard_nu_inner']:.2g}"
            elif s.hard._inner_model_type == 1:
                suptitl += f"dadt={s.PARS['hard_dadt_rchar']:.2g}"
            else:
                raise NotImplementedError
            plt.suptitle(suptitl)
        else:
            plt.suptitle(fname_extra)

    plt.tight_layout()
    
    if save:
        #plt.savefig('gwb_compare_ill_tng100_main.png',dpi=300)
        if fname_extra != '':
            fname = f'gwb_compare_nloud{NLOUD}_nreals{NREALS}_{fname_extra}.png'
        else:
            fname = f'gwb_compare_nloud{NLOUD}_nreals{NREALS}.png'            
        plt.savefig(f"{fpath}/{fname}", dpi=300)

    return

In [ ]:
def plot_loud_binary_params(freqs, sam_bgpars, sam_sspars, dpop, gpf_flag=0, lbl='', save=False):
    
    fig, axs = plt.subplots(nrows=2, ncols=3, sharex=True, figsize=[10,6])

    bg_alpha = 0.5
    ss_alpha = 0.2
    nskip = 1
    
    sam_bg_m1, sam_bg_m2 = utils.m1m2_from_mtmr(sam_bgpars[0,:,:], sam_bgpars[1,:,:])
    sam_ss_m1, sam_ss_m2 = utils.m1m2_from_mtmr(sam_sspars[0,:,:,:], sam_sspars[1,:,:,:])
    
    
    # 1st column panels
    axs[0,0].set(ylabel='Mtot',xscale='log',yscale='log') #xlabel='frequency', 
    # avg mt of bg sources for each freq for each real
    axs[0,0].plot(freqs, sam_bgpars[0,:,::nskip]/MSOL,'gray',lw=0.5,alpha=bg_alpha); 

    axs[1,0].set(ylabel='q',xlabel='frequency', xscale='log') #,yscale='log') #xlabel='frequency', 
    # avg mrat of bg sources for each freq for each real
    axs[1,0].plot(freqs, sam_bgpars[1,:,::nskip],'gray',lw=0.5,alpha=bg_alpha); 

    # 2nd column panels
    axs[0,1].set(ylabel='M1',xscale='log',yscale='log') #xlabel='frequency', 
    # avg mt of bg sources for each freq for each real
    axs[0,1].plot(freqs, sam_bg_m1[:, ::nskip]/MSOL,'gray',lw=0.5,alpha=bg_alpha); 
    
    axs[1,1].set(ylabel='M2',xlabel='frequency', xscale='log',yscale='log') #xlabel='frequency', 
    axs[1,1].plot(freqs, sam_bg_m2[:, ::nskip]/MSOL,'gray',lw=0.5,alpha=bg_alpha); # avg mt of bg sources for each freq for each real

    # 3rd column panels
    axs[0,2].set(ylabel='z_init',xscale='log', ylim=(0,2.5)) #,yscale='log')
    axs[0,2].plot(freqs, sam_bgpars[2,:,::nskip],'gray',lw=0.5,alpha=bg_alpha); # avg mrat of bg sources for each freq for each real

    axs[1,2].set(xlabel='frequency', ylabel='z_final',xscale='log', ylim=(0,2.5)) #,yscale='log')
    axs[1,2].plot(freqs, sam_bgpars[3,:,::nskip],'gray',lw=0.5,alpha=bg_alpha); # avg rzi of bg sources for each freq for each real

    if len(dpop) > 0:
        dpop_ss_m1, dpop_ss_m2 = utils.m1m2_from_mtmr(dpop.gwb.sspar[0], dpop.gwb.sspar[1])
        dpop_bg_m1, dpop_bg_m2 = utils.m1m2_from_mtmr(dpop.gwb.bgpar[0], dpop.gwb.bgpar[1])
        axs[0,0].plot(freqs, dpop.gwb.bgpar[0][:, ::nskip]/MSOL,'c',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real
        axs[1,0].plot(freqs, dpop.gwb.bgpar[1][:, ::nskip],'c',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real
        axs[0,1].plot(freqs, dpop_bg_m1[:, ::nskip]/MSOL,'c',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real
        axs[1,1].plot(freqs, dpop_bg_m2[:, ::nskip]/MSOL,'c',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real
        axs[0,2].plot(freqs, dpop.gwb.bgpar[2][:, ::nskip],'c',lw=0.2,alpha=bg_alpha); # avg mrat of bg sources for each freq for each real
        axs[1,2].plot(freqs, dpop.gwb.bgpar[3][:, ::nskip],'c',lw=0.2,alpha=bg_alpha); # avg rzi of bg sources for each freq for each real
        print(f"{dpop.gwb.bgpar[0].shape=}")
        print(dpop.gwb.sspar[3].shape) ##nfreq, nloud, nreals

    for ll in range(NLOUD):
        # 1st column panels
        # mt of loudest sources in each freq bin, avg over nreals
        axs[0,0].plot(freqs, np.mean(sam_sspars[0,:,:,ll],axis=1)/MSOL, 'k.', alpha=ss_alpha)

        # mrat of loudest sources in each freq bin, avg over nreals
        axs[1,0].plot(freqs, np.mean(sam_sspars[1,:,:,ll],axis=1), 'k.', alpha=ss_alpha) 

        # 2nd column panels
        # m1 of loudest sources in each freq bin, avg over nreals
        axs[0,1].plot(freqs, np.mean(sam_ss_m1[:,:,ll],axis=1)/MSOL, 'k.', alpha=ss_alpha)

        # m2 of loudest sources in each freq bin, avg over nreals
        axs[1,1].plot(freqs, np.mean(sam_ss_m2[:,:,ll],axis=1)/MSOL, 'k.', alpha=ss_alpha) 

        # 3rd column panels
        # rzi of loudest sources in each freq bin, avg over nreals
        axs[0,2].plot(freqs, np.mean(sam_sspars[2,:,:,ll],axis=1), 'k.', alpha=ss_alpha) 

        # rzf of loudest sources in each freq bin, avg over nreals
        sam_lbl = 'SAM' if ll==0 else None
        sim_lbl = 'sim' if ll==0 else None
        axs[1,2].plot(freqs, np.mean(sam_sspars[3,:,:,ll],axis=1), 'k.', alpha=ss_alpha, label=sam_lbl)

        if len(dpop)>0:
            axs[0,0].plot(dpop.freqs, np.mean(dpop.gwb.sspar[0][:,ll,:],axis=1)/MSOL, 'b.', alpha=ss_alpha) 
            axs[1,0].plot(dpop.freqs, np.mean(dpop.gwb.sspar[1][:,ll,:],axis=1), 'b.', alpha=ss_alpha)
            axs[0,1].plot(dpop.freqs, np.mean(dpop_ss_m1[:,ll,:],axis=1)/MSOL, 'b.', alpha=ss_alpha) 
            axs[1,1].plot(dpop.freqs, np.mean(dpop_ss_m2[:,ll,:],axis=1)/MSOL, 'b.', alpha=ss_alpha)
            axs[0,2].plot(dpop.freqs, np.mean(dpop.gwb.sspar[2][:,ll,:],axis=1), 'b.', alpha=ss_alpha) 
            axs[1,2].plot(dpop.freqs, np.mean(dpop.gwb.sspar[3][:,ll,:],axis=1), 'b.', alpha=ss_alpha, label=sim_lbl) 

    axs[1,2].legend(loc='upper left')
        
    #plt_title = dpop.lbl+' versus GPF SAM' if gpf_flag else dpop.lbl+' versus GMR SAM' 
    fig.suptitle(lbl)
    plt.tight_layout()
    #plt.subplots_adjust(hspace=0.1,wspace=0.3,top=0.92)

    if save: 
        fname_extra = 'gpf' if gpf_flag else 'gmr'
        fig.savefig(f'loud_bin_params_nloud{NLOUD}_{dpop.lbl}_{fname_extra}.png', dpi=300)


In [ ]:
def plot_evo(evo, freqs=None, sepa=None, ax=None, label=None, color=None, **kwargs):
    if (freqs is None) and (sepa is None):
        err = "Either `freqs` or `sepa` must be provided!"
        log.exception(err)
        raise ValueError(err)

    if freqs is not None:
        data = evo.at('fobs', freqs)
        xx = freqs * YR
        xlabel = 'GW Frequency [1/yr]'
    else:
        data = evo.at('sepa', sepa)
        xx = sepa / PC
        xlabel = 'Binary Separation [pc]'

    if ax is None:
        fig, ax = plot.figax(xlabel=xlabel)
    else:
        fig = ax.get_figure()

    def _draw_vals_conf(ax, xx, vals, color=color, label=label):
        if color is None:
            color = ax._get_lines.get_next_color()
        if label is not None:
            ax.set_ylabel(label, color=color)
            ax.tick_params(axis='y', which='both', colors=color)
        # vals = np.percentile(vals, [25, 50, 75], axis=0) / units
        vals = utils.quantiles(vals, [0.25, 0.50, 0.75], axis=0).T
        h1 = ax.fill_between(xx, vals[0], vals[-1], alpha=0.2, color=color)
        h2, = ax.plot(xx, vals[1], alpha=0.5, lw=2.0, color=color)
        return (h1, h2)

    # handles = []
    # labels = []

    name = 'Hardening Time [yr]'
    vals = np.fabs(data['sepa'] / data['dadt']) / YR
    _draw_vals_conf(ax, xx, vals, label=name)
    # handles.append(hh)
    # labels.append(name)

    # name = 'eccen'
    # tw = ax.twinx()
    # hh, nn = _draw_vals_conf(tw, freqs*YR, name, 'green')
    # if hh is not None:
    #     handles.append(hh)
    #     labels.append(nn)

    # ax.legend(handles, labels)
    return ax

In [ ]:
def plot_gwb_amps_vs_pars_6panel(sams, xvar='rgw9', fname_extra='', max_to_plot=4):

    fig, axs = plt.subplots(nrows=3, ncols=2, sharex=True, figsize=[6,4])

    yr_inv = np.array([10,3,1])
    freqs_to_plot = np.array([1/(y*YR) for y in yr_inv])
    print(f"{freqs_to_plot=}")
    A1_to_A10_PL_ratio = (freqs_to_plot[2]/freqs_to_plot[0])**(-2/3)
    A3_to_A10_PL_ratio = (freqs_to_plot[1]/freqs_to_plot[0])**(-2/3)
    A1_to_A3_PL_ratio = (freqs_to_plot[2]/freqs_to_plot[1])**(-2/3)
    
    for i in range(3): 
        axs[i,0].set_ylim(-16.0,-13.7)
        axs[i,1].set_ylim(0.0, 1.4)
    axs[0,0].set_ylabel(r'log$_{10}$ A$_{10yr}$')
    axs[1,0].set_ylabel(r'log$_{10}$ A$_{3yr}$')
    axs[2,0].set_ylabel(r'log$_{10}$ A$_{1yr}$')
    axs[0,1].set_ylabel(r'A$_{3yr}$ / A$_{10yr}$')
    axs[1,1].set_ylabel(r'A$_{1yr}$ / A$_{3yr}$')
    axs[2,1].set_ylabel(r'A$_{1yr}$ / A$_{10yr}$')

    cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples', 'GnBu',
                'RdPu', 'YlGnBu', 'YlGn','PuBuGn', 'OrRd', 'PuRd', 'YlOrRd', 'BuPu',
                'PuBu', 'YlGnBu_r']*5    
    #cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples',
    #            'Greys', 'RdPu', 'YlGnBu', 'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'BuPu',
    #            'GnBu', 'PuBu', 'PuBuGn', 'BuGn', 'YlGn']*10

    xvals = []
    hctot_list = []
    colors_list = []
    for n,s in enumerate(sams):

        if s.gwb_sam is None:
            continue

        #print(f"{n=} {cmap_arr[n]=}")
        cmap = plot._get_cmap(cmap_arr[n])
        color = cmap(0.7)
        #print(f"{color=}")
        colors_list.append(color)
        lw = np.arange(0.5,max_to_plot+1, 0.5)

        #mt, mr, = np.broadcast_arrays(
        #    s.sam.mtot[:, np.newaxis],
        #    s.sam.mrat[np.newaxis, :]
        #)

        if xvar=='tauin':
            #mt, mr, = np.broadcast_arrays(
            #    np.array([1.0e9*MSOL])[:, np.newaxis],
            #    np.array([1.0])[np.newaxis, :]
            #)
            tau_in_m9, _ = calc_total_tau_inner(s.hard, 1.0e9*MSOL, 1.0)
            xval = np.log10(tau_in_m9/YR) #np.log10(tau_in[:,0]/YR)
            xlbl=r'$\tau_{in}$'
        elif xvar=='rgw9':
            xval = np.log10(s.hard._r_gw_crit_9)
            xlbl = r'log$_{10}$(a$_{GW,9}$/R$_g$)'            
        elif xvar=='nui':
            xval = s.hard._nu_inner
            xlbl = r'$\nu_{inner}$'
        elif xvar=='alphgw':
            xval = s.hard._alpha_gw_crit
            xlbl = r'$\alpha_{GW}$'
        elif xvar=='betagw':
            xval = s.hard._beta_gw_crit
            xlbl = r'$\beta_{GW}$'
        elif xvar=='rch9':
            xval = np.log10(s.hard._rchar_9/PC)
            xlbl = r'log$_{10}$(a$_{char,9}$/PC)'
        elif xvar=='tout':
            xval = np.log10(s.hard._outer_time)
            xlbl = r'$\tau_{out}$'           
        else:
            raise ValueError()

        axs[2,0].set_xlabel(xlbl)
        
        xvals += [xval]
        
        freqs = s.PARS['freqs']

        hctot = calc_hctot(s.gwb_sam)
        hctot_list += [hctot]

    #print(f"{len(xvals)=} {xvals[0].size=}")
    xvals = np.array([xvals]).flatten()
    #print(f"{xvals.shape}")
    #print(f"{len(hctot_list)=} {hctot_list[0].shape=}")
    hctot_arr = np.array([hctot_list]).reshape(len(hctot_list), 
                                               hctot_list[0].shape[0], 
                                               hctot_list[0].shape[1])
    #print(f"{hctot_arr.shape}")

    #print(f"{len(colors_list)=}")
    #marker_colors = np.vstack([c[2] for c in colors_list])
    #marker_colors = [c[2] for c in colors_list]
    amps_list = []
    for i,f in enumerate(freqs_to_plot):

        idx = np.where(np.abs(freqs-f)==np.abs(freqs-f).min())[0]
        amps = hctot_arr[:,idx,:].reshape(hctot_arr.shape[0],hctot_arr.shape[2])
        #print(f"{amps.shape=}")
        #amps = hctot[idx,:].flatten()
        amps_list += [amps]
        __draw_med_conf(axs[i,0], xvals, np.log10(amps), fracs=[0.50], weights=None, plot={'color':'k'}, 
                        fill={}, filter=False, label=None, lw=1.0, ls='-')  
        #axs[i,0].plot(xvals, np.median(np.log10(amps),axis=1),'k-')
        axs[i,0].scatter(xvals, np.median(np.log10(amps),axis=1),color=colors_list, s=20, zorder=3)

    axs[0,1].plot([xvals.min(),xvals.max()],[A3_to_A10_PL_ratio,A3_to_A10_PL_ratio],'--',color='darkblue')
    __draw_med_conf(axs[0,1], xvals, amps_list[1]/amps_list[0], fracs=[0.50], weights=None, plot={'color':'k'}, 
                    fill={}, filter=False, label=None, lw=1.0, ls='-')  
    axs[1,1].plot([xvals.min(),xvals.max()],[A1_to_A3_PL_ratio,A1_to_A3_PL_ratio],'--',color='darkblue')
    __draw_med_conf(axs[1,1], xvals, amps_list[2]/amps_list[1], fracs=[0.50], weights=None, plot={'color':'k'}, 
                    fill={}, filter=False, label=None, lw=1.0, ls='-')  
    axs[2,1].plot([xvals.min(),xvals.max()],[A1_to_A10_PL_ratio,A1_to_A10_PL_ratio],'--',color='darkblue')
    __draw_med_conf(axs[2,1], xvals, amps_list[2]/amps_list[0], fracs=[0.50], weights=None, plot={'color':'k'}, 
                    fill={}, filter=False, label=None, lw=1.0, ls='-')  
    #axs[0,1].plot(xvals, np.median(amps_list[1]/amps_list[0],axis=1),'k-')
    #axs[1,1].plot(xvals, np.median(amps_list[2]/amps_list[1],axis=1),'k-') 
    #axs[2,1].plot(xvals, np.median(amps_list[2]/amps_list[0],axis=1),'k-') 
    axs[0,1].scatter(xvals, np.median(amps_list[1]/amps_list[0],axis=1), color=colors_list, s=20, zorder=3) 
    axs[1,1].scatter(xvals, np.median(amps_list[2]/amps_list[1],axis=1), color=colors_list, s=20, zorder=3) 
    axs[2,1].scatter(xvals, np.median(amps_list[2]/amps_list[0],axis=1), color=colors_list, s=20, zorder=3) 

    plt.tight_layout()



In [ ]:
def plot_tau_inner(sams, fname_extra='', xvar='rgw9', fid_value=2.5, max_q_to_plot=4,
                   NREALS=None, NLOUD=None, show_title=True, fpath='', horizontal=True,
                   cmap_arr=None, idx_to_plot=None, save=True):

    if horizontal:
        fig, axs = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=[7,3])
    else:
        fig, axs = plt.subplots(nrows=2, ncols=1, figsize=[4.5,5.5])
        
    if cmap_arr is None:
        cm = plot._get_cmap('viridis')
        colors = cm(np.linspace(0, 1, len(sams)))
        #cm = plot._get_cmap('tab20b')
        #cm2 = plot._get_cmap('tab20c')
        #cm3 = plot._get_cmap('tab20')
        #if len(sams)<=14:
        #    colors= np.vstack([cm(np.arange(0,20,4)),
        #                      cm2(np.arange(0,16,4)),
        #                      cm3(np.array([6,12,16,18,10]))]).reshape(14,4)
        #elif len(sams)>14 and len(sams)<=60:
        #    colors= np.vstack(cm(np.arange(20)),
        #                      cm2(np.arange(20)),
        #                      cm3(np.arange(20))).reshape(60,4)
        #else:
        #    raise ValueError()
        #cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples',
        #            'Greys', 'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
        #            'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']*10
        #cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples', 'GnBu',
        #            'RdPu', 'YlGnBu', 'YlGn','PuBuGn', 'OrRd', 'PuRd', 'YlOrRd', 'BuPu',
        #            'PuBu', 'YlGnBu_r']*5
    else:
        if len(cmap_arr) != len(sams):
            raise ValueError("len(cmap_arr) must match len(sams) if not None")
    colors_list = []

    #ax1 = fig.add_subplot(121)
    #ax2 = fig.add_subplot(122)

    xvals = []
    lgt7_list = []
    lgt9_list = []
    lgt11_list = []
    marker_list = []
    idx_fiducial = None
    idx_invalid = []
    if idx_to_plot is None:
        idx_to_plot = [0,len(sams)-1]
        
    for n,s in enumerate(sams):

        if s.gwb_sam is None:
            print(f"{n=} {s.gwb_sam=}")
            colors_list.append('gray')
            mrk='x'
            idx_invalid.append(n)
        else:
            if cmap_arr is not None:
                cm = plot._get_cmap(cmap_arr[n])
                #colors = cmap(np.linspace(0.3, 1, max_to_plot+1))
                colors_list.append(cm(0.7))
            else:
                colors_list.append(colors[n])
            mrk='o'
        marker_list.append(mrk)
        lw = np.arange(0.5,max_q_to_plot+1, 0.5)
        ls = [':','-','-.','--']
        
        mt, mr, = np.broadcast_arrays(
            s.sam.mtot[:, np.newaxis],
            s.sam.mrat[np.newaxis, :]
        )

        #print(s.sam.mtot/MSOL-1e7)
        ix7 = np.where(np.abs(s.sam.mtot/MSOL-1e7)==np.min(np.abs(s.sam.mtot/MSOL-1e7)))[0]
        ix9 = np.where(np.abs(s.sam.mtot/MSOL-1e9)==np.min(np.abs(s.sam.mtot/MSOL-1e9)))[0]
        ix11 = np.where(np.abs(s.sam.mtot/MSOL-1e11)==np.min(np.abs(s.sam.mtot/MSOL-1e11)))[0]
        #print(ix7, ix9, ix11)
        tau_in, rgw_crit = calc_total_tau_inner(s.hard, mt, mr)
        lgt7_list.append(np.log10(tau_in[ix7,-1]/YR))
        lgt9_list.append(np.log10(tau_in[ix9,-1]/YR))
        lgt11_list.append(np.log10(tau_in[ix11,-1]/YR))
        #print(f"{tauin7.shape=} {tauin7}")
        #print(f"{tauin9.shape=} {tauin9}")
        #print(f"{tauin11.shape=} {tauin11}")
        
        if xvar=='rgw9':
            xval=np.log10(s.hard._r_gw_crit_9)
            xlbl = r'log$_{\rm 10}$(a$_{\rm GW,9}/{\rm R_g}$)'            
        elif xvar=='nui':
            xval=s.hard._nu_inner
            xlbl = r'$\nu_{\rm inner}$'
        elif xvar=='alphgw':
            xval=s.hard._alpha_gw_crit
            xlbl = r'$\alpha_{\rm GW}$'
        elif xvar=='betagw':
            xval=s.hard._beta_gw_crit
            xlbl = r'$\beta_{\rm GW}$'
        elif xvar=='rch9':
            xval=np.log10(s.hard._rchar_9/PC)
            xlbl = r'log$_{\rm 10}$(a$_{\rm char,9}$/pc)'
        elif xvar=='tout':
            xval=np.log10(s.hard._outer_time/GYR)
            xlbl = r'$\tau_{\rm out}$ [Gyr]' 
        else:
            raise ValueError()

        xvals.append(xval)
        if np.abs(xval-fid_value) < 1.0e-6:
            idx_fiducial = n
            if n not in idx_to_plot:
                idx_to_plot.append(n)

        axs[0].set_xlabel(xlbl)
        axs[0].set_ylabel(r'log$_{\rm 10}$($\tau_{\rm in+gw}/{\rm Gyr}$)')
        axs[0].set_ylim(2,14)
        axs[1].set_xlabel(r'log$_{10}$(M/M$_{\odot}$)')
        if not horizontal:
            axs[1].set_ylabel(r'log$_{\rm 10}$($\tau_{\rm in+gw}/{\rm Gyr}$)')
        axs[1].set_ylim(2,14)
            
        #mt_nskip = int((s.sam.mtot.size-1)/(max_to_plot-1)) if s.sam.mtot.size>max_to_plot else 1
        mr_nskip = int((s.sam.mrat.size-1)/(max_q_to_plot-1)) if s.sam.mrat.size>max_q_to_plot else 1

        if n==0:
            axs[1].plot([np.log10(s.sam.mtot.min()/MSOL),np.log10(s.sam.mtot.max()/MSOL)],
                        [np.log10(13.7e9),np.log10(13.7e9)],'k--',lw=3)

        if n in idx_to_plot:
            j_plot=0
            for j in np.arange(0,s.sam.mrat.size,mr_nskip):
                _lw = 2.5 if n==idx_fiducial else 1.5
                _al = 1.0 #if n==idx_fiducial else 0.8
                axs[1].plot(np.log10(s.sam.mtot/MSOL), np.log10(tau_in[:,j]/YR),
                            alpha=_al,color=colors_list[n],lw=_lw,ls=ls[j_plot])
                j_plot += 1

        #print([xval], [np.log10(tauin7/YR)])
    axs[0].plot(xvals, lgt7_list,'k-',alpha=0.25, label=r"$10^7$")
    axs[0].plot(xvals, lgt9_list,'k-',alpha=0.5, label=r"$10^9$")
    axs[0].plot(xvals, lgt11_list,'k-',alpha=0.9, label=r"$10^{11}$")
    axs[0].plot([min(xvals),max(xvals)],[np.log10(13.7e9),np.log10(13.7e9)],'k--',lw=3)
    for n in range(len(xvals)):
        axs[0].scatter(xvals[n], lgt7_list[n], c=colors_list[n],s=15,marker=marker_list[n],zorder=3,alpha=0.5)
        axs[0].scatter(xvals[n], lgt9_list[n], c=colors_list[n],s=30,marker=marker_list[n],zorder=3,alpha=0.75)
        axs[0].scatter(xvals[n], lgt11_list[n], c=colors_list[n],s=45,marker=marker_list[n],zorder=3)
    if idx_fiducial is not None:
        axs[0].scatter(xvals[idx_fiducial], lgt7_list[idx_fiducial],facecolors='none', 
                    edgecolors=colors_list[idx_fiducial], s=75,marker=marker_list[idx_fiducial], zorder=3)
        axs[0].scatter(xvals[idx_fiducial], lgt9_list[idx_fiducial],facecolors='none', 
                    edgecolors=colors_list[idx_fiducial], s=90,marker=marker_list[idx_fiducial], zorder=3)
        axs[0].scatter(xvals[idx_fiducial], lgt11_list[idx_fiducial],facecolors='none', 
                    edgecolors=colors_list[idx_fiducial], s=105,marker=marker_list[idx_fiducial], zorder=3)

    axs[0].legend(title=r"$M$ [M$_{\odot}$], q=1")
    if show_title:
        plt.suptitle(fname_extra)
    plt.tight_layout()
    if horizontal:
        plt.subplots_adjust(wspace=0)
    else:
        plt.subplots_adjust(hspace=0.05)
        
    if save:
        if fname_extra != '':
            fname = f'tauingw_nloud{NLOUD}_nreals{NREALS}_{fname_extra}.png'
        else:
            fname = f'tauingw_nloud{NLOUD}_nreals{NREALS}.png'            
        plt.savefig(f"{fpath}/{fname}", dpi=300)


In [ ]:
def plot_gwb_amps_vs_pars_3panel(sams, xvar='rgw9', fid_value=2.5, fname_extra='', 
                                 NREALS=None, NLOUD=None, max_to_plot=4, fpath='', 
                                 cmap_arr=None, save=True):

    fig, axs = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=[4,5])

    yr_inv = np.array([10,3,1])
    freqs_to_plot = np.array([1/(y*YR) for y in yr_inv])
    print(f"{freqs_to_plot=}")
    A1_to_A10_PL_ratio = (freqs_to_plot[2]/freqs_to_plot[0])**(-2/3)
    A3_to_A10_PL_ratio = (freqs_to_plot[1]/freqs_to_plot[0])**(-2/3)
    A1_to_A3_PL_ratio = (freqs_to_plot[2]/freqs_to_plot[1])**(-2/3)
    
    axs[0].set_ylim(-16.3,-13.7)
    axs[1].set_ylim(-0.1, 2.1)
    axs[2].set_ylim(-0.1, 2.1)
    #axs[0,0].set_ylabel(r'log$_{10}$ A$_{10yr}$')
    #axs[1,0].set_ylabel(r'log$_{10}$ A$_{3yr}$')
    #axs[2,0].set_ylabel(r'log$_{10}$ A$_{1yr}$')
    axs[0].set_ylabel(r'log$_{\rm 10}$ $A(f_{\rm obs})$')
    axs[1].set_ylabel(r'$A_{3yr}$ / $A_{10yr}$')
    axs[2].set_ylabel(r'$A_{1yr}$ / $A_{10yr}$')

    if cmap_arr is None:
        cm = plot._get_cmap('viridis')
        colors = cm(np.linspace(0, 1, len(sams)))
        #cm = plot._get_cmap('tab20b')
        #cm2 = plot._get_cmap('tab20c')
        #cm3 = plot._get_cmap('tab20')
        #if len(sams)<=14:
        #    colors= np.vstack([cm(np.arange(0,20,4)),
        #                      cm2(np.arange(0,16,4)),
        #                      cm3(np.array([6,12,16,18,10]))]).reshape(14,4)
        #elif len(sams)>14 and len(sams)<=60:
        #    colors= np.vstack(cm(np.arange(20)),
        #                      cm2(np.arange(20)),
        #                      cm3(np.arange(20))).reshape(60,4)
        #else:
        #    raise ValueError()
        #cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples', 'GnBu',
        #            'RdPu', 'YlGnBu', 'YlGn','PuBuGn', 'OrRd', 'PuRd', 'YlOrRd', 'BuPu',
        #            'PuBu', 'YlGnBu_r']*5        
        #cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples',
        #            'Greys', 'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
        #            'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']*2
        #cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples', 'GnBu',
        #            'RdPu', 'YlGnBu', 'YlGn','PuBuGn', 'OrRd', 'PuRd', 'YlOrRd', 'BuPu',
        #            'PuBu', 'YlGnBu_r']*5
    else:
        if len(cmap_arr) != len(sams):
            raise ValueError("len(cmap_arr) must match len(sams) if not None")
    colors_list = []
 
    lw = np.arange(0.5,max_to_plot+1, 0.5)

    xvals = []
    xvals_invalid = []
    xvals_all = []
    hctot_list = []
    idx_invalid = np.array([])
    for n,s in enumerate(sams):

        if cmap_arr is not None:
            cm = plot._get_cmap(cmap_arr[n])
            #colors = cmap(np.linspace(0.3, 1, max_to_plot+1))
            colors_list.append(cm(0.7))
        else:
            colors_list.append(colors[n])

        if xvar=='tauin':
            tau_in_m9, _ = calc_total_tau_inner(s.hard, 1.0e9*MSOL, 1.0)
            xval = np.log10(tau_in_m9/YR) #np.log10(tau_in[:,0]/YR)
            xlbl=r'$\tau_{in}$'
        elif xvar=='rgw9':
            xval = np.log10(s.hard._r_gw_crit_9)
            xlbl = r'log$_{\rm 10}$(a$_{\rm GW,9}/{\rm R_g}$)'            
        elif xvar=='nui':
            xval = s.hard._nu_inner
            xlbl = r'$\nu_{\rm inner}$'
        elif xvar=='alphgw':
            xval = s.hard._alpha_gw_crit
            xlbl = r'$\alpha_{\rm GW}$'
        elif xvar=='betagw':
            xval = s.hard._beta_gw_crit
            xlbl = r'$\beta_{\rm GW}$'
        elif xvar=='rch9':
            xval = np.log10(s.hard._rchar_9/PC)
            xlbl = r'log$_{\rm 10}$(a$_{\rm char,9}$/pc)'
        elif xvar=='tout':
            xval = np.log10(s.hard._outer_time/GYR)
            xlbl = r'log$_{\rm 10}$(${\rm \tau_{out}/Gyr}$)'           
        else:
            raise ValueError()

        axs[2].set_xlabel(xlbl)

        print(f"{xval=} {fid_value=}")
        if np.abs(xval-fid_value) < 1.0e-6:
            idx_fiducial_all = n
            print(f"fiducial value {fid_value} has index {n} in list of all sams.")
        #else:
        #   print(f"{xval=} at index {n} is not fiducial value {fid_value}. sad!")

        freqs = s.PARS['freqs']
        
        if s.gwb_sam is None:
            xvals_invalid.append(xval)
            np.append(idx_invalid,n)
            hctot_list += [np.zeros((freqs.size,NREALS))*np.nan]
        else:
            xvals.append(xval)        
            hctot = calc_hctot(s.gwb_sam)
            print(f"{n=} {hctot.shape=}")
            hctot_list += [hctot]

        xvals_all.append(xval)
        

    xvals_invalid = np.array([xvals_invalid]).flatten()
    xvals_all = np.array([xvals_all]).flatten()
    xvals = np.array([xvals]).flatten()
    print(f"{xvals.shape=} {xvals_all.shape=}")
    idx_fiducial = np.where(xvals==xvals_all[idx_fiducial_all])[0][0]
    print(f"fiducial value {fid_value} has index {idx_fiducial} in list of valid sams.")

    hctot_arr = np.array([hctot_list]).reshape(len(hctot_list), 
                                               hctot_list[0].shape[0], 
                                               hctot_list[0].shape[1])
    amps_list = []
    for i,f in enumerate(freqs_to_plot):

        idx = np.where(np.abs(freqs-f)==np.abs(freqs-f).min())[0]
        amps = hctot_arr[:,idx,:].reshape(hctot_arr.shape[0],hctot_arr.shape[2])
        amps_list += [amps]
        __draw_med_conf(axs[0], xvals_all, np.log10(amps), fracs=[0.50], weights=None, plot={'color':'k'}, 
                        fill={}, filter=False, label=f"A(1/{yr_inv[i]}yr)", lw=2-0.75*i, ls='-')  
        med_amps = np.median(np.log10(amps),axis=1)
        #print(f"{amps[~idx_invalid,:].shape=} {med_amps.shape=} {xvals.shape=}")
        for n in range(len(xvals_all)):
            if xvals_all[n] not in xvals_invalid:
                axs[0].scatter(xvals_all[n], med_amps[n],
                               color=colors_list[n], s=20, zorder=3)
        axs[0].scatter(xvals_all[idx_fiducial_all], np.median(np.log10(amps[idx_fiducial_all,:])),
                       facecolors='none', edgecolors=colors_list[idx_fiducial_all], s=80, zorder=3)

    print(f"{amps_list[0].shape=} {amps_list[0].shape=}")
    axs[1].plot([xvals.min(),xvals.max()],[A3_to_A10_PL_ratio,A3_to_A10_PL_ratio],'--',color='darkblue')
    __draw_med_conf(axs[1], xvals_all, amps_list[1]/amps_list[0], fracs=[0.50], weights=None, plot={'color':'k'}, 
                    fill={}, filter=False, label=None, lw=1.0, ls='-')  
    axs[2].plot([xvals.min(),xvals.max()],[A1_to_A10_PL_ratio,A1_to_A10_PL_ratio],'--',color='darkblue',
                label=r"A $\propto$ f$^{-2/3}$")
    __draw_med_conf(axs[2], xvals_all, amps_list[2]/amps_list[0], fracs=[0.50], weights=None, plot={'color':'k'}, 
                    fill={}, filter=False, label=None, lw=1.0, ls='-')  
    axs[1].scatter(xvals_all[idx_fiducial_all], 
                   np.median(amps_list[1][idx_fiducial_all,:]/amps_list[0][idx_fiducial_all,:]), 
                   facecolors='none', edgecolors=colors_list[idx_fiducial_all], s=80, zorder=3) 
    axs[2].scatter(xvals_all[idx_fiducial_all], 
                   np.median(amps_list[2][idx_fiducial_all,:]/amps_list[0][idx_fiducial_all,:]), 
                   facecolors='none', edgecolors=colors_list[idx_fiducial_all], s=80, zorder=3) 

    ratio_1_0 = np.median(amps_list[1]/amps_list[0],axis=1)
    ratio_2_0 = np.median(amps_list[2]/amps_list[0],axis=1)
    for n in range(len(xvals_all)):
        if xvals_all[n] not in xvals_invalid:
            axs[1].scatter(xvals_all[n], ratio_1_0[n], color=colors_list[n], s=20, zorder=3) 
            axs[2].scatter(xvals_all[n], ratio_2_0[n], color=colors_list[n], s=20, zorder=3) 

    
    axs[0].legend()
    axs[2].legend()
    plt.tight_layout()
    plt.subplots_adjust(hspace=0)

    if save:
        if fname_extra != '':
            fname = f'gwb_amps_nloud{NLOUD}_nreals{NREALS}_{fname_extra}.png'
        else:
            fname = f'gwb_amps_nloud{NLOUD}_nreals{NREALS}.png'            
        plt.savefig(f"{fpath}/{fname}", dpi=300)



In [ ]:
def load_and_plot_newhard_sams(_subdir=None, _fname_type='new_hardening_type0_toutvar', 
                               _var_type='hard_outer_time', _fid_value=0.0,
                               dadt_idx_to_plot=None, _idx_fiducial=None,
                               NLOUD = 5, NREALS = 10, NFREQS = 40, num_steps=100,
                               plot_gwb_pars = False, _Tobs_yr = 20.0,
                               limit_alphgw_range = False,
                               skip_gwb_plot=False, skip_dadt_plot=False, skip_tauin_plot=False):

    if skip_gwb_plot and skip_dadt_plot and skip_tauin_plot:
        raise ValueError('nothing to plot. set flags to make at least one plot.')
        
    sams = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                              tau=None, data_dir=_SIM_MERGER_PATH, 
                              subdir=_subdir, fname_type=_fname_type)

    if limit_alphgw_range:
        print(f"initial length of list sams: {len(sams)=}")
        for i in range(len(sams)-1,-1,-1):
            if sams[i].hard._alpha_gw_crit < -0.5 or sams[i].hard._alpha_gw_crit > 0:
                print(f"eliminating alphagw={sams[i].hard._alpha_gw_crit} sam from list.")
                sams.pop(i)
            else:
                print(f"not eliminating alphagw={sams[i].hard._alpha_gw_crit} sam from list.")
                
        print(f"new: {len(sams)=}")
        
    gpf_flags = [0]*len(sams) 
    c_arr = 4*['g','c','m','b','k']

    all_cmaps = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples', 'GnBu',
                'RdPu', 'YlGnBu', 'YlGn','PuBuGn', 'OrRd', 'PuRd', 'YlOrRd', 'BuPu',
                'PuBu', 'YlGnBu_r']*5    
    #all_cmaps = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples',
    #            'Greys', 'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
    #            'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']*2
    
    _xvar = None
    
    for xv in ['rgw9','nui','alphgw','betagw','rch9','tout']:
        if xv in _fname_type and _xvar is None:
            _xvar = xv

    if not skip_dadt_plot:
        print(f"{dadt_idx_to_plot=}")
        dadt_list = []
        pars_list = []
        cmap_list = []
        _sublist_idx_fiducial=None
        for i,s in enumerate(sams): 

            if dadt_idx_to_plot is None or i in dadt_idx_to_plot:

                if i==_idx_fiducial:
                    _sublist_idx_fiducial = len(dadt_list)
                print(f"tout={s.hard._outer_time:.4g} "
                      f"rchar9={s.hard._rchar_9:.4g} alpha_char={s.hard._alpha_char:.4g} "
                      f"dadt_rchar={s.hard._dadt_rchar} nuin={s.hard._nu_inner} "
                      f"rgw9={s.hard._r_gw_crit_9:.4g} "
                      f"alpha_gw={s.hard._alpha_gw_crit:.4g} beta_gw={s.hard._beta_gw_crit:.4g}")

                tmp = calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                             nfreqs=NFREQS, num_steps=num_steps, verbose=False)
                dadt_list = dadt_list + [(tmp)]
                pars_list = pars_list + [s.PARS]
                cmap_list = cmap_list + [all_cmaps[i]]

        plot_dadt(dadt_list, pars_list, fixedTime='outer', gwcrit_units='pc', extra_panels=False,
                  fname_extra=_fname_type, var_name=_var_type, #cmap_arr=cmap_list,
                  fpath=f"{_SIM_MERGER_PATH}/{_subdir}",save=True,pubstyle=True, 
                  max_m_to_plot=3, max_q_to_plot=2, 
                  twopanel=True,
                  dadt_idx_to_plot=dadt_idx_to_plot, nsams_total=len(sams),
                  idx_fiducial=_sublist_idx_fiducial, Tobs_yr=_Tobs_yr)
        plot_dadt(dadt_list, pars_list, fixedTime='outer', gwcrit_units='rg', extra_panels=False,
                  fname_extra=_fname_type, var_name=_var_type, time_ylim=[0.3,1.5e12], #cmap_arr=cmap_list,
                  fpath=f"{_SIM_MERGER_PATH}/{_subdir}",save=True,pubstyle=True, 
                  max_m_to_plot=3, max_q_to_plot=2, 
                  twopanel=True,                  
                  dadt_idx_to_plot=dadt_idx_to_plot, nsams_total=len(sams),
                  idx_fiducial=_sublist_idx_fiducial, Tobs_yr=_Tobs_yr)

    if not skip_tauin_plot:
        plot_tau_inner(sams, fname_extra=_fname_type, xvar=_xvar, fid_value=_fid_value, max_q_to_plot=2,
                       NREALS=NREALS, NLOUD=NLOUD, show_title=False, idx_to_plot=dadt_idx_to_plot,
                       horizontal=False,
                       fpath=f"{_SIM_MERGER_PATH}/{_subdir}",save=True)

    if not skip_gwb_plot:
        compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, var_type=_var_type, fid_value=_fid_value, colors=c_arr,
                               lbl_extra=[f"{s.PARS[_var_type]:.2f}" for s in sams],
                               fname_extra=_fname_type, fpath=f"{_SIM_MERGER_PATH}/{_subdir}", 
                               save=True, show_title=False,
                               NLOUD=NLOUD, NREALS=NREALS)
            
        #plot_gwb_amps_vs_pars(sams, xvar='tauin', fname_extra=_fname_type)
        plot_gwb_amps_vs_pars_3panel(sams, xvar=_xvar, fid_value=_fid_value, fname_extra=_fname_type,
                                     NREALS=NREALS, NLOUD=NLOUD, 
                                     fpath=f"{_SIM_MERGER_PATH}/{_subdir}", save=True)

        if plot_gwb_pars:
            freqs, freqs_edges = utils.pta_freqs()
            for s in sams:         
                if s.gwb_sam is not None:
                    sam_hc_ss, sam_hc_bg, sam_sspar, sam_bgpar = s.gwb_sam
                    plot_loud_binary_params(freqs, sam_bgpar, sam_sspar, [], gpf_flag=0, 
                                            lbl=str(s.PARS[_var_type]), save=False)



In [ ]:
def cherry_pick_newhard_sams(plot_rgw=True, plot_stargas=True, plot_rgw_stargas=True):

    _fname_type = 'new_hardening_type0_rgw9var'
    _var_type='hard_r_gw_crit_9'
    #_fid_value=2.5
    NREALS=100
    NLOUD=5
    NFREQS=40
    num_steps=100
    c_arr = 4*['g','c','m','b','k']

    # ---- fiducial const-tin gets loaded either way:
    ctin_sams = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                   tau=None, data_dir=_SIM_MERGER_PATH, 
                                   subdir='gensams_nreal100/fid_const-tin', fname_type=_fname_type)
    for s in ctin_sams:
        if s.PARS[_var_type]==10.0**2.5:
            ctin_sam_to_plot = s
    ctin_dadt = calc_sam_dadt_from_pkl(ctin_sam_to_plot, nloud=NLOUD, nreals=NREALS, 
                                        nfreqs=NFREQS, num_steps=num_steps, verbose=False)

    # ---- fiducial const-rgw:
    if plot_rgw:
        crgw_sams = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                       tau=None, data_dir=_SIM_MERGER_PATH, 
                                       subdir='gensams_nreal100/const-rgw', fname_type=_fname_type)
        for s in crgw_sams:
            if s.PARS[_var_type]==10.0**2.5:
                crgw_sam_to_plot = s    
        crgw_dadt = calc_sam_dadt_from_pkl(crgw_sam_to_plot, nloud=NLOUD, nreals=NREALS, 
                                           nfreqs=NFREQS, num_steps=num_steps, verbose=False)

        plot_dadt([ctin_dadt,crgw_dadt], [ctin_sam_to_plot.PARS,crgw_sam_to_plot.PARS], 
                  fixedTime='outer', gwcrit_units='pc', extra_panels=False,
                  pubstyle=True, model_labels=['const-tin-nu0','const-agw-nu0'], Tobs_yr=20.0,
                  time_ylim=[0.02,3e11], cmap_arr = ['Blues','Oranges'], 
                  fname_extra=_fname_type+'_cherry_pick_fiducial', 
                  var_name=_var_type,fpath=f"{_SIM_MERGER_PATH}",save=True, 
                  max_m_to_plot=4, max_q_to_plot=4)
        plot_dadt([ctin_dadt,crgw_dadt], [ctin_sam_to_plot.PARS,crgw_sam_to_plot.PARS], 
                  fixedTime='outer', gwcrit_units='rg', extra_panels=False,
                  pubstyle=True, model_labels=['const-tin-nu0','const-agw-nu0'], Tobs_yr=20.0,
                  time_ylim=[0.02,3e11], cmap_arr = ['Blues','Oranges'], 
                  fname_extra=_fname_type+'_cherry_pick_fiducial', 
                  var_name=_var_type,fpath=f"{_SIM_MERGER_PATH}",save=True, 
                  max_m_to_plot=4, max_q_to_plot=4)

        compare_gwb_sim_vs_sam([ctin_sam_to_plot,crgw_sam_to_plot], [], gpf_flags=[0,0], var_type=_var_type, 
                               #fid_value=_fid_value, #colors=c_arr,
                               sam_lbls = [r"const-tin-nu0",r"const-agw-nu0"],
                               linestyles=['-','--'], cmap_arr = ['Blues','Oranges'], 
                               fname_extra=_fname_type+'_cherry_pick_fiducial', 
                               fpath=f"{_SIM_MERGER_PATH}", save=True, show_title=False, 
                               NLOUD=NLOUD, NREALS=NREALS)

    
    # ---- fiducial const-tin nu0, star, and gas:
    if plot_stargas:
        ctin_star_sams = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                       tau=None, data_dir=_SIM_MERGER_PATH, 
                                       subdir='gensams_nreal100/fid_const-tin_star', fname_type=_fname_type)
        for s in ctin_star_sams:
            if s.PARS[_var_type]==10.0**3.5:
                ctin_star_sam_to_plot = s
        ctin_star_dadt = calc_sam_dadt_from_pkl(ctin_star_sam_to_plot, nloud=NLOUD, nreals=NREALS, 
                                                nfreqs=NFREQS, num_steps=num_steps, verbose=False)

        ctin_gas_sams = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                      tau=None, data_dir=_SIM_MERGER_PATH, 
                                      subdir='gensams_nreal100/fid_const-tin_gas', fname_type=_fname_type)
        for s in ctin_gas_sams:
            if s.PARS[_var_type]==10.0**2:
                ctin_gas_sam_to_plot = s
        ctin_gas_dadt = calc_sam_dadt_from_pkl(ctin_gas_sam_to_plot, nloud=NLOUD, nreals=NREALS, 
                                               nfreqs=NFREQS, num_steps=num_steps, verbose=False)

        plot_dadt([ctin_dadt,ctin_star_dadt,ctin_gas_dadt], 
                  [ctin_sam_to_plot.PARS,ctin_star_sam_to_plot.PARS,ctin_gas_sam_to_plot.PARS], 
                  fixedTime='outer', gwcrit_units='rg', extra_panels=False, Tobs_yr=20.0,
                  cmap_arr = ['Blues','Greens', 'Purples'], rate_ylim=[1.1e-3,2e11],
                  pubstyle=True, model_labels=['const-tin-nu0','const-tin-star','const-tin-gas'],
                  fname_extra=_fname_type+'_cherry_pick_stargas', 
                  var_name=_var_type,fpath=f"{_SIM_MERGER_PATH}",save=True, 
                  max_m_to_plot=4, max_q_to_plot=4) 
        
        compare_gwb_sim_vs_sam([ctin_sam_to_plot,ctin_star_sam_to_plot, ctin_gas_sam_to_plot], 
                               [], gpf_flags=[0,0,0], var_type=_var_type, 
                               #fid_value=_fid_value, #colors=c_arr,
                               linestyles=['-','--','-.'],
                               sam_lbls = [r"const-tin-nu0",r"const-tin-star",r"const-tin-gas"],
                               cmap_arr = ['Blues','Greens', 'Purples'],
                               fname_extra=_fname_type+'_cherry_pick_stargas', 
                               fpath=f"{_SIM_MERGER_PATH}", save=True, show_title=False, 
                               NLOUD=NLOUD, NREALS=NREALS)

    # ---- fiducial const-rgw nu0, star, and gas:
    if plot_rgw_stargas and plot_rgw:
        crgw_star_sams = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                            tau=None, data_dir=_SIM_MERGER_PATH, 
                                            subdir='gensams_nreal100/const-rgw_star', fname_type=_fname_type)
        for s in crgw_star_sams:
            if s.PARS[_var_type]==10.0**3.5:
                crgw_star_sam_to_plot = s
        crgw_star_dadt = calc_sam_dadt_from_pkl(crgw_star_sam_to_plot, nloud=NLOUD, nreals=NREALS, 
                                                nfreqs=NFREQS, num_steps=num_steps, verbose=False)

        crgw_gas_sams = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                      tau=None, data_dir=_SIM_MERGER_PATH, 
                                      subdir='gensams_nreal100/const-rgw_gas', fname_type=_fname_type)
        for s in crgw_gas_sams:
            if s.PARS[_var_type]==10.0**2:
                crgw_gas_sam_to_plot = s
        crgw_gas_dadt = calc_sam_dadt_from_pkl(crgw_gas_sam_to_plot, nloud=NLOUD, nreals=NREALS, 
                                               nfreqs=NFREQS, num_steps=num_steps, verbose=False)

        plot_dadt([crgw_dadt,crgw_star_dadt,crgw_gas_dadt], 
                  [crgw_sam_to_plot.PARS,crgw_star_sam_to_plot.PARS,crgw_gas_sam_to_plot.PARS], 
                  fixedTime='outer', gwcrit_units='rg', extra_panels=False, Tobs_yr=20.0,
                  cmap_arr = ['Blues','Greens', 'Purples'], rate_ylim=[1.1e-3,1e11],
                  pubstyle=True, model_labels=['const-rgw-nu0','const-rgw-star','const-rgw-gas'],
                  fname_extra=_fname_type+'_cherry_pick_rgw_stargas', 
                  var_name=_var_type,fpath=f"{_SIM_MERGER_PATH}",save=True, 
                  max_m_to_plot=4, max_q_to_plot=4) 
        
        compare_gwb_sim_vs_sam([crgw_sam_to_plot,crgw_star_sam_to_plot, crgw_gas_sam_to_plot], 
                               [], gpf_flags=[0,0,0], var_type=_var_type, 
                               #fid_value=_fid_value, #colors=c_arr,
                               linestyles=['-','--','-.'],
                               sam_lbls = [r"const-rgw-nu0",r"const-rgw-star",r"const-rgw-gas"],
                               cmap_arr = ['Blues','Greens', 'Purples'],
                               fname_extra=_fname_type+'_cherry_pick_rgw_stargas', 
                               fpath=f"{_SIM_MERGER_PATH}", save=True, show_title=False, 
                               NLOUD=NLOUD, NREALS=NREALS)



In [ ]:
cherry_pick_newhard_sams(plot_stargas=False, plot_rgw_stargas=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/small_rgw9/fid_const-tin_gas', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2,
                           plot_gwb_pars=False,
                           NREALS=10,NLOUD=5,NFREQS=40,skip_dadt_plot=True,
                           #dadt_idx_to_plot=np.array([1,5,9]), 
                           _idx_fiducial=0,
                           skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/small_rgw9/const-rgw_gas', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2,
                           plot_gwb_pars=False,
                           NREALS=10,NLOUD=5,NFREQS=40,skip_dadt_plot=True,
                           #dadt_idx_to_plot=np.array([1,5,9]), 
                           _idx_fiducial=0,
                           skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2.5,
                           plot_gwb_pars=False,
                           NREALS=100,NLOUD=5,NFREQS=40,skip_dadt_plot=False,
                           dadt_idx_to_plot=np.array([1,5,10]), 
                           _idx_fiducial=5,
                           skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2.5,
                           plot_gwb_pars=False,
                           NREALS=100,NLOUD=5,NFREQS=40,skip_dadt_plot=False,
                           dadt_idx_to_plot= np.array([5,7,9]),
                           _idx_fiducial=5,                           
                           skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           plot_gwb_pars=False,
                           dadt_idx_to_plot= np.array([0,2,4]),  
                           _idx_fiducial=2, 
                           NREALS=100,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False,
                           limit_alphgw_range=True)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=+0.25,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=0,
                           plot_gwb_pars=False,
                           dadt_idx_to_plot=np.array([0,2,4,6]), 
                           _idx_fiducial=2,
                           NREALS=100,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=0.0,
                           plot_gwb_pars=False,
                           dadt_idx_to_plot=np.array([0,2,4,6]), 
                           _idx_fiducial=2,
                           NREALS=100,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_star', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=3.5,
                           plot_gwb_pars=False,
                           #dadt_idx_to_plot=np.array([0,2,4,6]), 
                           #_idx_fiducial=0,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
#load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
#                           _fname_type='new_hardening_type0_rgw9var', 
#                           _var_type='hard_r_gw_crit_9',_fid_value=2.5,
#                           plot_gwb_pars=False,
#                           NREALS=100,skip_dadt_plot=False,
#                           #dadt_idx_to_plot=np.arange(0,10,2),
#                           skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=+0.25,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False,
                           limit_alphgw_range=True)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_gas', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False,
                           limit_alphgw_range=True)
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_star', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False,
                           limit_alphgw_range=True)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_gas', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False,
                           limit_alphgw_range=True)
load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_star', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False,
                           limit_alphgw_range=True)

In [ ]:
#load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw', 
#                           _fname_type='new_hardening_type0_rgw9var', 
#                           _var_type='hard_r_gw_crit_9',_fid_value=2.5,
#                           plot_gwb_pars=False,
#                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_star', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=3.5,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_star', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_star', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=+0.25,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_star', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=-1,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_star', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_star', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_gas', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_gas', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_gas', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=+0.25,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_gas', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=2,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_gas', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/fid_const-tin_gas', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_star', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=3.5,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_star', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_star', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_star', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=-1,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_star', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_star', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_gas', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_gas', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_gas', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_gas', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=2,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_gas', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_nreal100/const-rgw_gas', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

# shape = 200

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/shape200/fid_const-tin', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2.5,
                           plot_gwb_pars=False,
                           NREALS=100,NLOUD=5,NFREQS=40,skip_dadt_plot=True,
                           dadt_idx_to_plot=np.array([1,5,9]), 
                           _idx_fiducial=5,
                           skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/shape200/fid_const-tin', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           plot_gwb_pars=False,
                           NREALS=100,NLOUD=5,NFREQS=40,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/shape200/fid_const-tin', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=0.0,
                           plot_gwb_pars=False,
                           NREALS=100,NLOUD=5,NFREQS=40,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/large_rgw9/const-rgw', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=4,
                           plot_gwb_pars=False,
                           NREALS=10,NLOUD=5,NFREQS=40,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/large_rgw9/const-rgw_star', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=4,
                           plot_gwb_pars=False,
                           NREALS=10,NLOUD=5,NFREQS=40,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/small_rgw9/const-rgw_gas', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=1.0,
                           plot_gwb_pars=False,
                           NREALS=10,NLOUD=5,NFREQS=40,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/small_rgw9/const-rgw_gas', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=1.0,
                           plot_gwb_pars=False,
                           NREALS=10,NLOUD=5,NFREQS=40,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/small_rgw9/fid_const-tin_gas', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=1.0,
                           plot_gwb_pars=False,
                           NREALS=10,NLOUD=5,NFREQS=40,skip_dadt_plot=True,
                           skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange', 
                           _fname_type='new_hardening_type0_alphgwvar_M6-7', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange', 
                           _fname_type='new_hardening_type0_alphgwvar_M7-8', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange', 
                           _fname_type='new_hardening_type0_alphgwvar_M8-9', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange', 
                           _fname_type='new_hardening_type0_alphgwvar_M9-10', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange', 
                           _fname_type='new_hardening_type0_alphgwvar_M10-11', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange', 
                           _fname_type='new_hardening_type0_alphgwvar_M11-12', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange', 
                           _fname_type='new_hardening_type0_alphgwvar_M4-5', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange', 
                           _fname_type='new_hardening_type0_alphgwvar_M5-6', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange_shape12', 
                           _fname_type='new_hardening_type0_alphgwvar_M4-5', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange_shape12', 
                           _fname_type='new_hardening_type0_alphgwvar_M5-6', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange_shape12', 
                           _fname_type='new_hardening_type0_alphgwvar_M6-7', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange_shape12', 
                           _fname_type='new_hardening_type0_alphgwvar_M7-8', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange_shape12', 
                           _fname_type='new_hardening_type0_alphgwvar_M8-9', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange_shape12', 
                           _fname_type='new_hardening_type0_alphgwvar_M9-10', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange_shape12', 
                           _fname_type='new_hardening_type0_alphgwvar_M10-11', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange_shape12', 
                           _fname_type='new_hardening_type0_alphgwvar_M11-12', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange', 
                           _fname_type='new_hardening_type0_alphgwvar_M6-10', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/limited_mtrange', 
                           _fname_type='new_hardening_type0_alphgwvar_fine', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/test_nogwphase_gas', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2.5,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/test_nogwphase_gas', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
#load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts', 
load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/fiducial_const-tin', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2.5,
                           plot_gwb_pars=True,
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/fiducial_const-tin', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           plot_gwb_pars=True,
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/fiducial_const-tin', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=0.25,
                           plot_gwb_pars=True,                           
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/fiducial_const-tin', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=0,
                           plot_gwb_pars=True,                           
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/fiducial_const-tin', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=0.0,
                           plot_gwb_pars=True,                           
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/fiducial_const-tin', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=0.0,
                           plot_gwb_pars=True,
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
#_tout_dflt = 1.0 # Gyr
#_nui_dflt = 0.0
#_rgw9_dflt = 10.0**2.5 # Rg
#_alphgw_dflt = 0.0
#_betagw_dflt = 0.0
#_rch9_dflt = 1.0 # pc
#_alphch_dflt = -2/3 # not varying

load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/const-rgw', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2.5,
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/const-rgw', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/const-rgw', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/const-rgw', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/const-rgw', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=1.0,
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_rpta_alphchn23/const-rgw', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=False,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
#_nui_dflt = -1.0
#_rgw9_dflt = 10.0**3.5 # Rg
#_alphgw_dflt = -0.25
#_betagw_dflt = +0.25
#_alphch_dflt = -2/3 # not varying
#_rch9_dflt = 1.0 # pc
#_tout_dflt = 1.0 # Gyr

sub = 'gensams_test/new_rpta_alphchn23/fiducial_const-tin_star'

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=3.5,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=+0.25,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=-1.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=1.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
#_tout_dflt = 1.0 # Gyr
#_nui_dflt = +2.0
#_rgw9_dflt = 10.0**2 # Rg
#_alphgw_dflt = -0.25
#_betagw_dflt = +0.25
#_rch9_dflt = 1.0 # pc
#_alphch_dflt = -2/3 # not varying

sub = 'gensams_test/new_rpta_alphchn23/fiducial_const-tin_gas'


load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=+0.25,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=+2.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=1.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=1.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
#_tout_dflt = 1.0 # Gyr
#_nui_dflt = 0.0
#_rgw9_dflt = 10.0**3 # Rg
#_alphgw_dflt = -0.25
#_betagw_dflt = +0.25
#_rch9_dflt = 1.0 # pc
#_alphch_dflt = -2/3 # not varying

sub = 'gensams_test/new_rpta_alphchn23/fiducial_const-tin_rgw91e3'


load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=3.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=+0.25,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=1.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir=sub, _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
#_tout_dflt = 1.0 # Gyr
#_nui_dflt = -1.0
#_rgw9_dflt = 10.0**3.5 # Rg
#_alphgw_dflt = 0
#_betagw_dflt = 0
#_rch9_dflt = 1.0 # pc
#_alphch_dflt = 0 # not varying


load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_star', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=3.5,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_star', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_star', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_star', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=-1.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_star', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=1.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_star', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=1.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
#_tout_dflt = 1.0 # Gyr
#_nui_dflt = +2.0
#_rgw9_dflt = 10.0**2 # Rg
#_alphgw_dflt = 0
#_betagw_dflt = 0
#_rch9_dflt = 1.0 # pc
#_alphch_dflt = 0 # not varying


load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_gas', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_gas', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_gas', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=0.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_gas', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=+2.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_gas', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=1.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

load_and_plot_newhard_sams(_subdir='gensams_test/new_no_rgw_cuts/alpha0_const-rgw_gas', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=1.0,
                           NREALS=10,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

#### fake production runs i ended up changing more stuff after these 'PRODUCTION RUNS'

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fiducial_beta0pt25_devdflt_mod0_nui0', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2.5,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fiducial_devdflt_mod0_nui0', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',_fid_value=2.5,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fiducial_devdflt_mod0_nui0', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',_fid_value=0.0,
                           NREALS=100,
                           skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fiducial_devdflt_mod0_nui0', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',_fid_value=-0.25,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fiducial_devdflt_mod0_nui0', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit',_fid_value=0,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fiducial_devdflt_mod0_nui0', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',_fid_value=0.5,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_nreal100/fiducial_devdflt_mod0_nui0', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',_fid_value=1.0,
                           NREALS=100,skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
##### alpha_char = -1/2, evolving mmbulge amplitude

load_and_plot_newhard_sams(_subdir='gensams_060426/kh13evol_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/kh13evol_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/kh13evol_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/kh13evol_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/kh13evol_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/devdflt_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=True)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=True)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=True)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=True)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=True)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=True)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=True)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=True)

In [ ]:
# alpha_char = -1/2

load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

#load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
#                           _fname_type='new_hardening_type0_alphgwvar', 
#                           _var_type='hard_alpha_gw_crit',plot_dadt=False,plot_gwb=False)
#load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
#                           _fname_type='new_hardening_type0_alphgwvar', 
#                           _var_type='hard_alpha_gw_crit',plot_dadt=False,plot_gwb=False)


In [ ]:
# alpha_char = -2/3

load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)



In [ ]:
# alpha_char = 0

load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
# alpha_char = -0.5, star-like
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)


In [ ]:
# alpha_char = -2/3, star-like

load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)



In [ ]:
# alpha_char = 0, star-like

load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_nomassdep', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_nomassdep', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_nomassdep', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_nomassdep', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
# alpha_char = -0.5, gas-like
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',plot_dadt=False,plot_gwb=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',plot_dadt=False,plot_gwb=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',plot_dadt=False,plot_gwb=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',plot_dadt=False,plot_gwb=False)


In [ ]:
# alpha_char = -2/3, gas-like

load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)



In [ ]:
# alpha_char = 0, star-like

load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_nomassdep', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_nomassdep', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_nomassdep', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_nomassdep', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',skip_dadt_plot=True,skip_gwb_plot=False,skip_tauin_plot=False)

In [ ]:
__M = 1e9*MSOL
__a = 1.0*PC
__tauGW_Gyr = 5 * (__a)**4 * SPLC**5 / (64*NWTG**3 * __M**3 *0.25) / GYR
__f_em = np.sqrt(NWTG*__M) / (__a**1.5 * np.pi)
_f_em = 2*utils.kepler_freq_from_sepa(__M, __a)
print(__tauGW_Gyr)
print(__f_em, __f_em*YR, _f_em, _f_em*YR)
print(8156*4 * (__f_em*YR)**(-8/3) / 1e9)
print(8156*4 * (_f_em*YR)**(-8/3) / 1e9)

In [ ]:
9300.725953923618/2325.1606663657294

In [ ]:
5*SPLC**5 / (64*(NWTG*__M)**(5/3)*0.25*np.pi**(8/3)) *YR**(8/3) / YR

In [ ]:
8156*4

# load SAMs

# Old hardening model

In [ ]:
def load_and_plot_oldhard_sams(_subdir=None, _fname_type='ph15_ng15', tau=None, 
                               NLOUD = 5, NREALS = 10, NFREQS = 40, num_steps=100,
                               gwb_only=False):

    sams = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS,
                              tau=tau, data_dir=_SIM_MERGER_PATH, 
                              subdir=_subdir, fname_type=_fname_type)

    c_arr = 4*['g','c','m','b','k']
    gpf_flags = [1]*len(sams) 
    _var_type = None
    
    if not gwb_only:
        dadt_list = []
        pars_list = []
        for s in sams: 

            tmp = calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                         nfreqs=NFREQS, num_steps=num_steps, verbose=False)
            #log.info(f"{tmp[1]._outer_time=}")
            dadt_list = dadt_list + [(tmp)]
            pars_list = pars_list + [s.PARS]

        
        
        #log.info(len(dadt_list), len(dadt_list[0]))
        #log.info(f"{dadt_list[0][1]._outer_time=}")
        plot_dadt(dadt_list, pars_list, fixedTime='total', extra_panels=False, 
                  fname_extra=_fname_type, var_name=_var_type) #, max_to_plot=3)
        plot_dadt(dadt_list, pars_list, fixedTime='total', gwcrit_units='rg', extra_panels=False,
                  fname_extra=_fname_type, var_name=_var_type) #, max_to_plot=3)

    if _var_type is None:
        lbl_extra = ''
    else: lbl_extra = [str(s.PARS[_var_type]) for s in sams]
        
    compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, #colors=c_arr,
                           lbl_extra=lbl_extra, fname_extra='compare_gwb'+_fname_type, 
                           save=True, NLOUD=NLOUD, NREALS=NREALS)
  

In [ ]:
load_and_plot_oldhard_sams(_subdir='gensams_ng15', _fname_type='ph15_ng15', tau=0.1)

In [ ]:
load_and_plot_oldhard_sams(_subdir='gensams_ng15', _fname_type='ph15_ng15', tau=0.5)

In [ ]:
load_and_plot_oldhard_sams(_subdir='gensams_ng15', _fname_type='ph15_ng15', tau=1.0)

In [ ]:
load_and_plot_oldhard_sams(_subdir='gensams_ng15', _fname_type='ph15_ng15', tau=2.0)

## Model 0, defaults: $r_{\rm gw,9}=10^{2.5}R_g$, $\nu_{\rm in}=0$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -0.5$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values
## NG15 vals for non-hardening pars

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time') #plot_dadt=False,plot_gwb=False)

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## star-like Model 0, defaults: $r_{\rm gw,9}=10^{3.5}R_g$, $\nu_{\rm in}=-1$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -0.5$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values
## NG15 vals for non-hardening pars

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## gas-like Model 0, defaults: $r_{\rm gw,9}=10^{2}R_g$, $\nu_{\rm in}=2$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -0.5$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values
## NG15 vals for non-hardening pars

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_alphach-0pt5', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## no-mass-dependence Model 0, defaults: $r_{\rm gw,9}=10^{2.5}R_g$, $\nu_{\rm in}=0$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = 0$, $\alpha_{gw}=0$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values
## NG15 vals for non-hardening pars

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui0_nomassdep', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## star-like no-mass-dependence Model 0, defaults: $r_{\rm gw,9}=10^{3.5}R_g$, $\nu_{\rm in}=-1$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -0.5$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values
## NG15 vals for non-hardening pars

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_nomassdep', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_nomassdep', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_nomassdep', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_nomassdep', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_nomassdep', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui-1_nomassdep', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## gas-like no-mass-dependence Model 0, defaults: $r_{\rm gw,9}=10^{2}R_g$, $\nu_{\rm in}=2$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -0.5$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values
## NG15 vals for non-hardening pars

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_nomassdep', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_nomassdep', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_nomassdep', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_nomassdep', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_nomassdep', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_060426/ng15_mod0_nui2_nomassdep', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## Model 0, defaults: $r_{\rm gw,9}=10^{2.5}R_g$, $\nu_{\rm in}=0$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -2/3$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values
## version 1: dev defaults for non-hardening pars
## version 2: NG15 vals for non-hardening pars

In [ ]:
# version 1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

# version 2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9', gwb_only=True)


# version 3
load_and_plot_newhard_sams(_subdir='gensams_test', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit', gwb_only=True)

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',gwb_only=True)

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui0_rgw91e2pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',gwb_only=True)

## 'gas-like' model
## Model 0, defaults: $r_{\rm gw,9}=10^{2}R_g$, $\nu_{\rm in}=+2$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -2/3$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',gwb_only=True)

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',gwb_only=True)

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit', gwb_only=True)

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',gwb_only=True)

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui2_rgw91e2_newalph_rchar', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',gwb_only=True)

## 'star-like' model
## Model 0, defaults: $r_{\rm gw,9}=10^{3.5}R_g$, $\nu_{\rm in}=-1$,  $r_{\rm char,9} = 3.0$pc, $\alpha_{\rm char} = -2/3$
(note: rch9=3pc is about the minimum value it can have and still have rch>rgw for M=1e12 msun when rgw9=10^3.5Rg and alpha=-0.25.)
#### testing the new defaults for rchar9 and alpha_char, with lower varied values of rchar9 and a larger range of alpha_gw values

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time',gwb_only=True)

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9',gwb_only=True)

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit',gwb_only=True)

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner',gwb_only=True)

In [ ]:
#v1
load_and_plot_newhard_sams(_subdir='gensams_v052126/mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')
#v2
load_and_plot_newhard_sams(_subdir='gensams_v052126/ng15_mod0_nui-1_rgw91e3pt5_newalph_rchar', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9',gwb_only=True)

## rgw9 default = 10^2.5 Rg, nu_inner default = 0

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## Model 0 w/ rchar9: rgw9 default = 10^3 Rg, nu_inner default = 0

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e3_rch91e2/', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e3_rch91e2/', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e3_rch91e2/', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e3_rch91e2/', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e3_rch91e2/', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## Model 0 w/ rchar9: rgw9 default = 10^2 Rg, nu_inner default = 0

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e2_rch91e2/', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e2_rch91e2/', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e2_rch91e2/', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e2_rch91e2/', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui0_alphgw025_rgw91e2_rch91e2/', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## Model 0 w/ rchar9: rgw9 default = 10^2.5 Rg, nu_inner default = 0.5

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui05_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui05_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui05_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui05_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui05_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## Model 0 w/ rchar9: rgw9 default = 10^2.5 Rg, nu_inner default = -0.5

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui-05_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui-05_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui-05_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_alphgwvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui-05_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_rgw9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050626/mod0_tout1_nui-05_alphgw025_rgw91e2pt5_rch91e2/', 
                           _fname_type='new_hardening_type0_rch9var', 
                           _var_type='hard_rchar_9')

## testing beta_gw_crit

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v051526/mod0_tout1_nui0_alphgw025_rgw91e2pt5_rch91e2', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v051526/mod0_tout1_nui0_alphgw025_rgw91e3_rch91e2', 
                           _fname_type='new_hardening_type0_betagwvar', 
                           _var_type='hard_beta_gw_crit')

# Model1

# Param sweep with default r9 = 10^3.5 Rg

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e3pt5_rch1e2', 
                           _fname_type='new_hardening_type1_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e3pt5_rch1e2', 
                           _fname_type='new_hardening_type1_dadtvar', 
                           _var_type='hard_dadt_rchar')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e3pt5_rch1e2', 
                           _fname_type='new_hardening_type1_r9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e3pt5_rch1e2', 
                           _fname_type='new_hardening_type1_alphvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e3pt5_rch1e2', 
                           _fname_type='new_hardening_type1_rchvar', 
                           _var_type='hard_rchar')

# Param sweep with r9 = 1000 Rg

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e3_rch1e2', 
                           _fname_type='new_hardening_type1_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e3_rch1e2', 
                           _fname_type='new_hardening_type1_dadtvar', 
                           _var_type='hard_dadt_rchar')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e3_rch1e2', 
                           _fname_type='new_hardening_type1_r9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e3_rch1e2', 
                           _fname_type='new_hardening_type1_alphvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e3_rch1e2', 
                           _fname_type='new_hardening_type1_rchvar', 
                           _var_type='hard_rchar')

# Param sweep with r9 = 10^2.5 Rg

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e2pt5_rch1e2', 
                           _fname_type='new_hardening_type1_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e2pt5_rch1e2', 
                           _fname_type='new_hardening_type1_dadtvar', 
                           _var_type='hard_dadt_rchar')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e2pt5_rch1e2', 
                           _fname_type='new_hardening_type1_r9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e2pt5_rch1e2', 
                           _fname_type='new_hardening_type1_alphvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e2pt5_rch1e2', 
                           _fname_type='new_hardening_type1_rchvar', 
                           _var_type='hard_rchar')

# Param sweep with r9 = 100 Rg

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e2_rch1e2', 
                           _fname_type='new_hardening_type1_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e2_rch1e2', 
                           _fname_type='new_hardening_type1_dadtvar', 
                           _var_type='hard_dadt_rchar')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e2_rch1e2', 
                           _fname_type='new_hardening_type1_r9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e2_rch1e2', 
                           _fname_type='new_hardening_type1_alphvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_model1only_v042126/mod1_tout1_dadtn1e6_alph025_r91e2_rch1e2', 
                           _fname_type='new_hardening_type1_rchvar', 
                           _var_type='hard_rchar')

# Model 0

## Model 0: Param sweep with default r9 = 10^3.5 Rg

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e3pt5_rch1e2', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e3pt5_rch1e2', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e3pt5_rch1e2', 
                           _fname_type='new_hardening_type0_r9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e3pt5_rch1e2', 
                           _fname_type='new_hardening_type0_alphvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e3pt5_rch1e2', 
                           _fname_type='new_hardening_type0_rchvar', 
                           _var_type='hard_rchar')

## Model 0: Param sweep with default r9 = 10^3 Rg

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e3_rch1e2', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e3_rch1e2', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e3_rch1e2', 
                           _fname_type='new_hardening_type0_r9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e3_rch1e2', 
                           _fname_type='new_hardening_type0_alphvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e3_rch1e2', 
                           _fname_type='new_hardening_type0_rchvar', 
                           _var_type='hard_rchar')

## Model 0:  Param sweep with default r9 = 10^2.5 Rg

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e2pt5_rch1e2', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e2pt5_rch1e2', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e2pt5_rch1e2', 
                           _fname_type='new_hardening_type0_r9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e2pt5_rch1e2', 
                           _fname_type='new_hardening_type0_alphvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e2pt5_rch1e2', 
                           _fname_type='new_hardening_type0_rchvar', 
                           _var_type='hard_rchar')

## Model 0: Param sweep with r9 = 10^2 Rg

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e2_rch1e2', 
                           _fname_type='new_hardening_type0_toutvar', 
                           _var_type='hard_outer_time')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e2_rch1e2', 
                           _fname_type='new_hardening_type0_nuivar', 
                           _var_type='hard_nu_inner')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e2_rch1e2', 
                           _fname_type='new_hardening_type0_r9var', 
                           _var_type='hard_r_gw_crit_9')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e2_rch1e2', 
                           _fname_type='new_hardening_type0_alphvar', 
                           _var_type='hard_alpha_gw_crit')

In [ ]:
load_and_plot_newhard_sams(_subdir='gensams_v050126/mod0_tout1_nui0_alph025_r91e2_rch1e2', 
                           _fname_type='new_hardening_type0_rchvar', 
                           _var_type='hard_rchar')

# older stuff

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
num_steps=100
sams_newhard_typ1_toutvar = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                               tau=None, data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type1_toutvar')

dadt_list = []
for s in sams_newhard_typ1_toutvar: 
    print(s.model_type)
    print(s.PARS['hard_outer_time'])
    print(f"pars: {s.PARS['hard_rchar']=} {s.PARS['hard_dadt_rchar']=} {s.PARS['hard_r_gw_crit_9']=} {s.PARS['hard_alpha_gw_crit']=}")
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps, verbose=False))]

print([f"tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ1_toutvar])
gpf_flags = [0]*len(sams_newhard_typ1_toutvar) 

compare_gwb_sim_vs_sam(sams_newhard_typ1_toutvar, [], gpf_flags=gpf_flags, colors=['g','c','m'],
                       lbl_extra=[f" tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ1_toutvar],
                       fname_extra='compare_gwb_newhard_typ1_toutvar', save=True)

plot_dadt(dadt_list, fixedTime='outer', extra_panels=False) #, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg') #, max_to_plot=3)


In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
num_steps=100
sams_newhard_typ1_dadtvar = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                               tau=None, data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type1_dadtvar')

dadt_list = []
for s in sams_newhard_typ1_dadtvar: 
    print(s.model_type)
    print(s.PARS['hard_outer_time'])
    print(f"pars: {s.PARS['hard_rchar']=} {s.PARS['hard_dadt_rchar']=} {s.PARS['hard_r_gw_crit_9']=} {s.PARS['hard_alpha_gw_crit']=}")
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps, verbose=True))]

print([f"tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ1_dadtvar])
gpf_flags = [0]*len(sams_newhard_typ1_dadtvar) 

compare_gwb_sim_vs_sam(sams_newhard_typ1_dadtvar, [], gpf_flags=gpf_flags, colors=['g','c','m,'b','k'],
                       lbl_extra=[f" dadtrc={s.PARS['hard_dadt_rchar']}" for s in sams_newhard_typ1_dadtvar],
                       fname_extra='compare_gwb_newhard_typ1_dadtvar', save=True)

plot_dadt(dadt_list, fixedTime='outer', extra_panels=False) #, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg') #, max_to_plot=3)


In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
num_steps=100
sams_newhard_typ1_r9var = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                               tau=None, data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type1_r9var')

dadt_list = []
for s in sams_newhard_typ1_r9var: 
    print(s.model_type)
    print(s.PARS['hard_outer_time'])
    print(f"pars: {s.PARS['hard_rchar']=} {s.PARS['hard_dadt_rchar']=} {s.PARS['hard_r_gw_crit_9']=} {s.PARS['hard_alpha_gw_crit']=}")
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps, verbose=False))]

print([f"tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ1_r9var])
gpf_flags = [0]*len(sams_newhard_typ1_r9var) 

compare_gwb_sim_vs_sam(sams_newhard_typ1_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m,'b','k'],
                       lbl_extra=[f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ1_r9var],
                       fname_extra='compare_gwb_newhard_typ1_r9var', save=True)

plot_dadt(dadt_list, fixedTime='outer', extra_panels=False) #, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg') #, max_to_plot=3)


In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
num_steps=100
sams_newhard_typ1_alphvar = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                               tau=None, data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type1_alphvar')

dadt_list = []
for s in sams_newhard_typ1_alphvar: 
    print(s.model_type)
    print(s.PARS['hard_outer_time'])
    print(f"pars: {s.PARS['hard_rchar']=} {s.PARS['hard_dadt_rchar']=} {s.PARS['hard_r_gw_crit_9']=} {s.PARS['hard_alpha_gw_crit']=}")
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps, verbose=True))]

print([f"tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ1_alphvar])
gpf_flags = [0]*len(sams_newhard_typ1_alphvar) 

compare_gwb_sim_vs_sam(sams_newhard_typ1_alphvar, [], gpf_flags=gpf_flags, colors=['g','c','m,'b','k'],
                       lbl_extra=[rf" $\alpha$={s.PARS['hard_alpha_gw_crit']}" for s in sams_newhard_typ1_alphvar],
                       fname_extra='compare_gwb_newhard_typ1_r9var', save=True)

plot_dadt(dadt_list, fixedTime='outer', extra_panels=False) #, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg') #, max_to_plot=3)


In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
num_steps=100
sams_newhard_typ1_toutvar = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                               tau=None, data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type1_toutvar')

dadt_list = []
for s in sams_newhard_typ1_toutvar: 
    print(s.model_type)
    print(s.PARS['hard_outer_time'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]

print([f"tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ1_toutvar])
gpf_flags = [0]*len(sams_newhard_typ1_toutvar) 

compare_gwb_sim_vs_sam(sams_newhard_typ1_toutvar, [], gpf_flags=gpf_flags, colors=['g','c','m'],
                       lbl_extra=[f" tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ0_toutvar],
                       fname_extra='compare_gwb_newhard_typ1_toutvar', save=True)

plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)


In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
num_steps=100
sams_newhard_typ1_toutvar = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                               tau=None, data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type1_toutvar')

dadt_list = []
for s in sams_newhard_typ1_toutvar: 
    print(s.model_type)
    print(s.PARS['hard_outer_time'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]

print([f"tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ1_toutvar])
gpf_flags = [0]*len(sams_newhard_typ1_toutvar) 

compare_gwb_sim_vs_sam(sams_newhard_typ1_toutvar, [], gpf_flags=gpf_flags, colors=['g','c','m'],
                       lbl_extra=[f" tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ0_toutvar],
                       fname_extra='compare_gwb_newhard_typ1_toutvar', save=False)

plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)


# New hardening model type 0

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
num_steps=100
sams_newhard_typ0_toutvar = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False,
                                               tau=None, data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_toutvar')

dadt_list = []
for s in sams_newhard_typ0_toutvar: 
    print(s.model_type)
    print(s.PARS['hard_outer_time'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]

print([f"tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ0_toutvar])
gpf_flags = [0]*len(sams_newhard_typ0_toutvar) 

#gwb_amps(sams=sams_newhard_typ0_toutvar, dpops=None, fname='compare_amps_newhard_typ0_toutvar', 
#         gpf_flags=gpf_flags, colors=['g','c','m'],
#         lbl_extra=[f" tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ0_toutvar])

compare_gwb_sim_vs_sam(sams_newhard_typ0_toutvar, [], gpf_flags=gpf_flags, colors=['g','c','m'],
                       lbl_extra=[f" tout={s.PARS['hard_outer_time']}" for s in sams_newhard_typ0_toutvar],
                       fname_extra='compare_gwb_newhard_typ0_toutvar', save=False)

plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)


In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 100
NFREQS = 40
sams_newhard_typ0_nuivar = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_nuivar')

dadt_list = []
for s in sams_newhard_typ0_nuivar: 
    print(s.model_type)
    print(s.PARS['hard_gamma_inner'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]

print([f"nui={s.PARS['hard_gamma_inner']}" for s in sams_newhard_typ0_nuivar])
_lbl_extra = [f" nui={s.PARS['hard_gamma_inner']}" for s in sams_newhard_typ0_nuivar]
gpf_flags = [0]*len(sams_newhard_typ0_nuivar) 

#gwb_amps(sams=sams_newhard_typ0_nuivar, dpops=None, fname='compare_amps_newhard_typ0_nuivar', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_nuivar, [], gpf_flags=gpf_flags, colors=['g','c','m','b'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_nuivar', save=False)

plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)


In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var', save=False)

plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)


In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_alphvar = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_alphvar')
dadt_list=[]
for s in sams_newhard_typ0_alphvar: 
    print(s.model_type)
    print(s.PARS['hard_alpha_gw_crit'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" alph={s.PARS['hard_alpha_gw_crit']}" for s in sams_newhard_typ0_alphvar]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_alphvar) 

#gwb_amps(sams=sams_newhard_typ0_alphvar, dpops=None, fname='compare_amps_newhard_typ0_alphvar', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_alphvar, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_alphvar', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui-1alph-025')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui-15alph-025')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui-15alph-025', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui0alph-025')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    print(np.sqrt( np.sum(s.gwb_sam[0]**2,axis=2) + s.gwb_sam[1]**2 ))
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui0alph-025', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui-1alph-0')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui-1alph-0', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui-1alph-05')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui-1alph-05', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui-15alph-05')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui-15alph-05', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui-15alph0')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui-15alph0', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui0alph-05')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui0alph-05', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui0alph0')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui0alph0', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui-05alph-05')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui-05alph-05', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui-05alph-025')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui-05alph-025', save=False)

In [ ]:
# ---- Load new hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 10
NFREQS = 40
sams_newhard_typ0_r9var = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                                          data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_r9var_nui-05alph0')

dadt_list=[]
for s in sams_newhard_typ0_r9var: 
    print(s.model_type)
    print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_newhard_typ0_r9var) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_newhard_typ0_r9var, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_r9var_nui-05alph0', save=False)

In [ ]:
# ---- Load new hardening SAMs
NLOUD = 5
NREALS = 10
NFREQS = 40
par_lbl = 'hard_alpha_gw_crit'
ext_name = 'alphvarnui-1r9100'
sams = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                     data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_'+ext_name)
dadt_list=[]
for s in sams: 
    print(s.model_type)
    print(s.PARS[par_lbl])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" alph={s.PARS[par_lbl]}" for s in sams]
print(_lbl_extra)
gpf_flags = [0]*len(sams) 
compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_'+ext_name, save=False)

In [ ]:
# ---- Load new hardening SAMs
NLOUD = 5
NREALS = 10
NFREQS = 40
par_lbl = 'hard_alpha_gw_crit'
ext_name = 'alphvarnui0r91e3'
sams = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                     data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_'+ext_name)
dadt_list=[]
for s in sams: 
    print(s.model_type)
    print(s.PARS[par_lbl])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)
    
_lbl_extra = [f" alph={s.PARS[par_lbl]}" for s in sams]
print(_lbl_extra)
gpf_flags = [0]*len(sams) 
compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_'+ext_name, save=False)

In [ ]:
# ---- Load new hardening SAMs
NLOUD = 5
NREALS = 10
NFREQS = 40
par_lbl = 'hard_alpha_gw_crit'
ext_name = 'alphvarnui0r9100'
sams = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                     data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_'+ext_name)
dadt_list=[]
for s in sams: 
    print(s.model_type)
    print(s.PARS[par_lbl])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" alph={s.PARS[par_lbl]}" for s in sams]
print(_lbl_extra)
gpf_flags = [0]*len(sams) 
compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_'+ext_name, save=False)

In [ ]:
# ---- Load new hardening SAMs
NLOUD = 5
NREALS = 10
NFREQS = 40
par_lbl = 'hard_alpha_gw_crit'
ext_name = 'alphvarnui-15r91e3'
sams = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                     data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_'+ext_name)
dadt_list=[]
for s in sams: 
    print(s.model_type)
    print(s.PARS[par_lbl])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)


_lbl_extra = [f" alph={s.PARS[par_lbl]}" for s in sams]
print(_lbl_extra)
gpf_flags = [0]*len(sams) 
compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_'+ext_name, save=False)

In [ ]:
# ---- Load new hardening SAMs
NLOUD = 5
NREALS = 10
NFREQS = 40
par_lbl = 'hard_gamma_inner'
ext_name = 'nuivaralph-025r9300'
sams = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                     data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_'+ext_name)
dadt_list=[]
for s in sams: 
    print(s.model_type)
    print(s.PARS[par_lbl])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" nui={s.PARS[par_lbl]}" for s in sams]
print(_lbl_extra)
gpf_flags = [0]*len(sams) 
compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, colors=['g','c','m','b'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_'+ext_name, save=False)

In [ ]:
# ---- Load new hardening SAMs
NLOUD = 5
NREALS = 10
NFREQS = 40
par_lbl = 'hard_gamma_inner'
ext_name = 'nuivaralph-025r9100'
sams = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                     data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_'+ext_name)
dadt_list=[]
for s in sams: 
    print(s.model_type)
    print(s.PARS[par_lbl])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" nui={s.PARS[par_lbl]}" for s in sams]
print(_lbl_extra)
gpf_flags = [0]*len(sams) 
compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, colors=['g','c','m','b'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_'+ext_name, save=False)

In [ ]:
# ---- Load new hardening SAMs
NLOUD = 5
NREALS = 10
NFREQS = 40
par_lbl = 'hard_gamma_inner'
ext_name = 'nuivaralph-025r930'
sams = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                     data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_'+ext_name)
dadt_list=[]
for s in sams: 
    print(s.model_type)
    print(s.PARS[par_lbl])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" nui={s.PARS[par_lbl]}" for s in sams]
print(_lbl_extra)
gpf_flags = [0]*len(sams) 
compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, colors=['g','c','m','b'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_'+ext_name, save=False)

In [ ]:
# ---- Load new hardening SAMs
NLOUD = 5
NREALS = 10
NFREQS = 40
par_lbl = 'hard_gamma_inner'
ext_name = 'nuivaralph-05r9100'
sams = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                     data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_'+ext_name)
dadt_list=[]
for s in sams: 
    print(s.model_type)
    print(s.PARS[par_lbl])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = [f" nui={s.PARS[par_lbl]}" for s in sams]
print(_lbl_extra)
gpf_flags = [0]*len(sams) 
compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, colors=['g','c','m','b'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_'+ext_name, save=False)

In [ ]:
# ---- Load new hardening SAMs
NLOUD = 5
NREALS = 10
NFREQS = 40
par_lbl = 'hard_gamma_inner'
ext_name = 'nuivaralph0r9100'
sams = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=None,
                     data_dir=_SIM_MERGER_PATH, fname_type='new_hardening_type0_'+ext_name)
dadt_list=[]
for s in sams: 
    print(s.model_type)
    print(s.PARS[par_lbl])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]

_lbl_extra = [f" nui={s.PARS[par_lbl]}" for s in sams]
print(_lbl_extra)
gpf_flags = [0]*len(sams) 
compare_gwb_sim_vs_sam(sams, [], gpf_flags=gpf_flags, colors=['g','c','m','b'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_newhard_typ0_'+ext_name, save=False)

plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='outer', extra_panels=False, distance_units='rg', max_to_plot=3)


In [ ]:
# ---- Load old hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 50
NFREQS = 40
sams_tau555 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=5.55,
                            data_dir=_SIM_MERGER_PATH, fname_type='old_new_mods_compare')


dadt_list=[]
for s in sams_tau555: 
    print(s.model_type)
    #print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='total', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='total', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = "" ###[f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_tau555) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_tau555, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_old_new_mods', save=False)

In [ ]:
# ---- Load old hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 50
NFREQS = 40
sams_tau01 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=0.1,
                            data_dir=_SIM_MERGER_PATH, fname_type='old_new_mods_compare')


dadt_list=[]
for s in sams_tau01: 
    print(s.model_type)
    #print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='total', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='total', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = "" ###[f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_tau01) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_tau01, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='compare_gwb_old_new_mods', save=False)

In [ ]:
# ---- Load old hardening SAMs

# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 500
NFREQS = 40
sams_mm_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, gpf_flag=False, tau=1.0,
                            data_dir=_SIM_MERGER_PATH, fname_type='manual_moddefs')


dadt_list=[]
for s in sams_mm_tau1: 
    print(s.model_type)
    #print(s.PARS['hard_r_gw_crit_9'])
    dadt_list = dadt_list + [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                     nfreqs=NFREQS, num_steps=num_steps))]
plot_dadt(dadt_list, fixedTime='total', extra_panels=False, max_to_plot=3)
plot_dadt(dadt_list, fixedTime='total', extra_panels=False, distance_units='rg', max_to_plot=3)

_lbl_extra = "" ###[f" r9={s.PARS['hard_r_gw_crit_9']}" for s in sams_newhard_typ0_r9var]
print(_lbl_extra)
gpf_flags = [0]*len(sams_mm_tau1) 

#gwb_amps(sams=sams_newhard_typ0_r9var, dpops=None, fname='compare_amps_newhard_typ0_r9var', 
#         gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
#         lbl_extra=_lbl_extra)

compare_gwb_sim_vs_sam(sams_mm_tau1, [], gpf_flags=gpf_flags, colors=['g','c','m','b','k'],
                       lbl_extra=_lbl_extra,
                       fname_extra='manual_moddefs_tau1', save=False)

In [ ]:

# ---- Load SAMs
sams_gmr_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
                              gpf_flag=False, tau=1.0, data_dir=_SIM_MERGER_PATH)
#sams_gmr_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=False, tau=3.0, data_dir=_PATH_DATA)
#sams_gpf_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=1.0, data_dir=_PATH_DATA)
#sams_gpf_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=3.0, data_dir=_PATH_DATA)

for s in sams_gmr_tau1: #+sams_gmr_tau3+sams_gpf_tau1+sams_gpf_tau3:
    print(s.model_type)
    sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
#sam_tau_list = [1.0, 3.0]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
gmr_sams_list = [ s for s in sams_gmr_tau1 #+sams_gmr_tau3
                  if (s.model_type in sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]
#gpf_sams_list = [ s for s in sams_gpf_tau1+sams_gpf_tau3
#                  if (s.model_type in sam_mod_list and 
#                  s.PARS['hard_time'] in sam_tau_list) ]

for s in gmr_sams_list:
    print(s.model_type)
sams_list = gmr_sams_list 
gpf_flags = [0]*len(gmr_sams_list) 
#sams_list = gmr_sams_list + gpf_sams_list
#gpf_flags = [0]*len(gmr_sams_list) + [1]*len(gpf_sams_list) 

#gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_gpf_tau1', color_hardmods=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau1', save=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau3', save=False)

# ---- GMR TAU=1
gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_tau1', 
         color_hardmods=False, gpf_flags=gpf_flags)

compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
                       fname_extra='compare_gwb_gmr_tau1', save=False)
## ---- GMR TAU=3
#gwb_amps(sams=sams_gmr_tau3, dpops=None, fname='compare_amps_gmr_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gmr_tau3, [], gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='compare_gwb_gmr_tau3', save=False)
## ---- GPF TAU=1
#gwb_amps(sams=sams_gpf_tau1, dpops=None, fname='compare_amps_gpf_tau1', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau1, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau1', save=False)
## ---- GPF TAU=3
#gwb_amps(sams=sams_gpf_tau3, dpops=None, fname='compare_amps_gpf_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau3, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau3', save=False)


In [ ]:

# ---- Load SAMs
sams_old_new_mods_compare = load_sam_data(nloud=NLOUD, nreals=50, nfreqs=NFREQS, 
                                          gpf_flag=None, tau=None, data_dir=_SIM_MERGER_PATH,
                                          fname_type='old_new_mods_compare')
#sams_gmr_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=False, tau=3.0, data_dir=_PATH_DATA)
#sams_gpf_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=1.0, data_dir=_PATH_DATA)
#sams_gpf_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=3.0, data_dir=_PATH_DATA)

for s in sams_old_new_mods_compare: #+sams_gmr_tau3+sams_gpf_tau1+sams_gpf_tau3:
    print(s.model_type)

gpf_flags = [1, 0]

gwb_amps(sams=sams_old_new_mods_compare, dpops=None, fname='compare_amps_sams_old_new_mods_compare', 
         color_hardmods=False, gpf_flags=gpf_flags)

compare_gwb_sim_vs_sam(sams_old_new_mods_compare, [], gpf_flags=gpf_flags, 
                       fname_extra='compare_gwb__sams_old_new_mods_compare', save=False)


In [ ]:

# ---- Load SAMs
sams_old_new_mods_compare = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
                                          gpf_flag=None, tau=1.0, data_dir=_SIM_MERGER_PATH)
#sams_gmr_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=False, tau=3.0, data_dir=_PATH_DATA)
#sams_gpf_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=1.0, data_dir=_PATH_DATA)
#sams_gpf_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=3.0, data_dir=_PATH_DATA)

for s in sams_old_new_mods_compare: #+sams_gmr_tau3+sams_gpf_tau1+sams_gpf_tau3:
    print(s.model_type)

sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
#sam_tau_list = [1.0, 3.0]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
gmr_sams_list = [ s for s in sams_gmr_tau1 #+sams_gmr_tau3
                  if (s.model_type in sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]
#gpf_sams_list = [ s for s in sams_gpf_tau1+sams_gpf_tau3
#                  if (s.model_type in sam_mod_list and 
#                  s.PARS['hard_time'] in sam_tau_list) ]

for s in gmr_sams_list:
    print(s.model_type)
sams_list = gmr_sams_list 
gpf_flags = [0]*len(gmr_sams_list) 
#sams_list = gmr_sams_list + gpf_sams_list
#gpf_flags = [0]*len(gmr_sams_list) + [1]*len(gpf_sams_list) 

#gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_gpf_tau1', color_hardmods=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau1', save=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau3', save=False)

# ---- GMR TAU=1
gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_tau1', 
         color_hardmods=False, gpf_flags=gpf_flags)

compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
                       fname_extra='compare_gwb_gmr_tau1', save=False)
## ---- GMR TAU=3
#gwb_amps(sams=sams_gmr_tau3, dpops=None, fname='compare_amps_gmr_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gmr_tau3, [], gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='compare_gwb_gmr_tau3', save=False)
## ---- GPF TAU=1
#gwb_amps(sams=sams_gpf_tau1, dpops=None, fname='compare_amps_gpf_tau1', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau1, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau1', save=False)
## ---- GPF TAU=3
#gwb_amps(sams=sams_gpf_tau3, dpops=None, fname='compare_amps_gpf_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau3, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau3', save=False)


# OLD STUFF (copied from `compare_gwb_discrete_vs_sam.ipynb`)

In [ ]:
print(len(gwb_sam_gpf))
print(len(gwb_sam_gpf[0]), len(gwb_sam_gpf[1]), 
      len(gwb_sam_gpf[2]), len(gwb_sam_gpf[3])) #, len(gwb_sam_new))
print(gwb_sam_gpf[0].shape) # hcss: [nfreqs, nreals, nloudest]
print(gwb_sam_gpf[1].shape) # hcbg: [nfreqs, nreals]
print(gwb_sam_gpf[2].shape) # sspars: [nsspars, nfreqs, nreals, nloudest]
print(gwb_sam_gpf[3].shape) # bgpars: [nbgpars, nfreqs, nreals]
gwb_sam_gpf_hcss = gwb_sam_gpf[0]
gwb_sam_gpf_hcbg = gwb_sam_gpf[1]
gwb_sam_gpf_hctot = np.sqrt( np.sum(gwb_sam_gpf[0]**2,axis=2) + gwb_sam_gpf[1]**2 )
gwb_sam_gpf_sspars = gwb_sam_gpf[2]
gwb_sam_gpf_bgpars = gwb_sam_gpf[3]

print(gwb_sam_gpf_hcss.shape) # hcss: [nfreqs, nreals, nloudest]
print(gwb_sam_gpf_hcbg.shape) # hcbg: [nfreqs, nreals]
print(gwb_sam_gpf_hctot.shape) # hctot: [nfreqs, nreals]
print(gwb_sam_gpf_sspars.shape) # sspars: [nsspars, nfreqs, nreals, nloudest]
print(gwb_sam_gpf_bgpars.shape) # bgpars: [nbgpars, nfreqs, nreals]



gwb_sam_gmr_hcss = gwb_sam_gmr[0]
gwb_sam_gmr_hcbg = gwb_sam_gmr[1]
gwb_sam_gmr_hctot = np.sqrt( np.sum(gwb_sam_gmr[0]**2,axis=2) + gwb_sam_gmr[1]**2 )
gwb_sam_gmr_sspars = gwb_sam_gmr[2]
gwb_sam_gmr_bgpars = gwb_sam_gmr[3]



# sspars:
# sspar[0,ff,rr,ll] = mt[mm]
# sspar[1,ff,rr,ll] = mr[qq]
# sspar[2,ff,rr,ll] = rz[zz]
# sspar[3,ff,rr,ll] = redz_final[mm,qq,zz,ff]
# bgpars: 
# bgpar[0,ff,rr] = m_bg/sum_bg # bg avg mass
# bgpar[1,ff,rr] = q_bg/sum_bg # bg avg ratio
# bgpar[2,ff,rr] = z_bg/sum_bg # bg avg redshift
# bgpar[3,ff,rr] = zfinal_bg/sum_bg # bg avg redshift after hardening
# bgpar[4,ff,rr] = dcom_bg/sum_bg # bg avg comoving distance after hardening
# bgpar[5,ff,rr] = sepa_bg/sum_bg # bg avg binary separation after hardening
# bgpar[6,ff,rr] = angs_bg/sum_bg # bg avg binary angular separation after hardening


In [ ]:
plot_loud_binary_params(freqs_gpf, gwb_sam_gpf_bgpars, gwb_sam_gpf_sspars, all_fsa_dpops[0], gpf_flag=1, save=True)
#plot_loud_binary_params(freqs_gmr, gwb_sam_gmr_bgpars, gwb_sam_gmr_sspars, all_fsa_dpops[0], save=True)

In [ ]:
compare_gwb_sim_vs_sam(freqs_gpf, [gwb_new_sam_gpf], all_dpops, fsa_dpops=None, outfilename='gwb_compare_all_fid.png')

In [ ]:
fig = holo.plot.plot_gwb(freqs_gpf, gwb_new_sam_gpf)

In [ ]:
fig = holo.plot.plot_gwb(freqs_gpf, gwb_sam_gpf_hctot)

In [ ]:
fig = holo.plot.plot_gwb(freqs_gmr, gwb_new_sam_gmr)

In [ ]:
fig = holo.plot.plot_gwb(freqs_gmr, gwb_sam_gmr_hctot)

In [ ]:
# Calculate the total lifetime of each binary
ncols = 2
nrows = 3
fig, axes = plot.figax(scale='lin', xlabel='Time: actual/specified', ylabel='density',figsize=(8,5),
                       ncols=ncols, nrows=nrows, wspace=0.3,hspace=0.3)
#times = [evo_old_ill.tlook, evo_new_ill.tlook, evo_tng100_1.tlook, evo_tng300_1.tlook, hard.tlook]
times = [dpop_tng100_1.evo.tlook, dpop_tng300_1.evo.tlook] #, hard.tlook]
print(len(times))
i = 0
for idx,ax in np.ndenumerate(axes):
    i = idx[0]*ncols + idx[1]
    if i >= len(times): break
    print(times[i].shape)
    print(f"idx: {idx}, i: {i}")
    dt = times[i][:, 0] - times[i][:, -1]
    # Create figure
    # use kalepy to plot distribution
    kale.dist1d(dt/tau, density=True, ax=ax)
    #ax.legend()
    
plt.show()

In [ ]:
def Mc(M,q):
    return M * q**0.6 / (1+q)**1.2

In [ ]:
#qarr = np.logspace(-5,0,100)
#Marr = np.logspace(4,12,100)
qarr = np.logspace(-2,0,100)
Marr = np.logspace(8,11,100)
#print(qarr)
plt.xscale('log')
plt.yscale('log')
plt.tick_params(axis='y', which='both', left=True, right=True)
plt.plot(qarr,Mc(10**8,qarr))
plt.plot(qarr,Mc(10**9,qarr))
plt.plot(qarr,Mc(10**10,qarr))
plt.plot(qarr,10**10*qarr**0.6)

In [ ]:
plt.xscale('log')
plt.yscale('log')
plt.plot(Marr,Mc(Marr,0.1))
plt.plot(Marr,Mc(Marr,0.5))
plt.plot(Marr,Mc(Marr,1.0))

In [ ]:
plt.xscale('log')
#plt.yscale('log')
plt.plot(Marr,(Mc(Marr,1.0)-Mc(Marr,0.1))/Mc(Marr,0.1))
plt.plot(Marr,(Mc(Marr,1.0)-Mc(Marr,0.01))/Mc(Marr,0.01))
#plt.plot(Marr,Mc(Marr,0.5))
#plt.plot(Marr,Mc(Marr,1.0))

In [ ]:
Mc(1e9,1.0)/Mc(1e9,0.01)

In [ ]:
cm = plot._get_cmap('tab20')
cm2 = plot._get_cmap('tab20b')
#colors= np.append(cm(np.arange(20)),cm2(np.arange(20))).reshape(40,4)
colors= np.append(cm(np.arange(0,20,2)),cm2(np.arange(0,20,2))).reshape(20,4)
print(colors.shape)

plt.scatter(np.arange(20),np.arange(20),color=colors)